In [3]:
import pandas as pd
from pathlib import Path

# Ensure data directories and paths are established
data_dir = Path("data/raw")
file_path_1 = data_dir / "customer_orders.parquet"
file_path_2 = data_dir / "pickup_point_orders.parquet"
file_path_3 = data_dir / "product_reference.parquet"
file_path_4 = data_dir / "product_snapshots.parquet"
file_path_5 = data_dir / "product_top.parquet"

# Load datasets
df_cust_ord = pd.read_parquet(file_path_1)
df_pup_ord = pd.read_parquet(file_path_2)
df_prod_ref = pd.read_parquet(file_path_3)
df_prod_snap = pd.read_parquet(file_path_4)
df_prod_top = pd.read_parquet(file_path_5)
df_pickup = pd.read_parquet(file_path_2)


In [4]:
import pandas as pd

# ============================================================
# 1. Extract distinct SKU-EAN combinations
# ============================================================

df_customer = df_cust_ord.copy()

customer_pairs = (
    df_customer[["sku", "ean_code"]]
    .dropna(subset=["sku", "ean_code"])
    .drop_duplicates()
)

snap_pairs = (
    df_prod_snap[["sku", "ean_code"]]
    .dropna(subset=["sku", "ean_code"])
    .drop_duplicates()
)


# ============================================================
# 2. Identify SKUs with multiple EANs WITHIN each dataset
# ============================================================

customer_ean_counts = (
    customer_pairs.groupby("sku")["ean_code"].nunique()
)

snap_ean_counts = (
    snap_pairs.groupby("sku")["ean_code"].nunique()
)

multi_ean_skus_customer = customer_ean_counts[
    customer_ean_counts > 1
].index

multi_ean_skus_snap = snap_ean_counts[
    snap_ean_counts > 1
].index


# ============================================================
# 3. Display multi-EAN SKUs and their EAN lists
# ============================================================

df_multi_ean_customer = (
    customer_pairs[
        customer_pairs["sku"].isin(multi_ean_skus_customer)
    ]
    .groupby("sku")["ean_code"]
    .agg(list)
    .reset_index(name="customer_eans")
)

df_multi_ean_snap = (
    snap_pairs[
        snap_pairs["sku"].isin(multi_ean_skus_snap)
    ]
    .groupby("sku")["ean_code"]
    .agg(list)
    .reset_index(name="snapshot_eans")
)

print("=== MULTIPLE EANS PER SKU ===")
print(f"Customer: {len(df_multi_ean_customer)} SKUs")
print(f"Snapshots: {len(df_multi_ean_snap)} SKUs")

display(df_multi_ean_customer)
display(df_multi_ean_snap)


# ============================================================
# 4. Compare EAN SETS for each SKU across datasets
# ============================================================

customer_mapping = (
    customer_pairs
    .groupby("sku")["ean_code"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="customer_eans")
)

snap_mapping = (
    snap_pairs
    .groupby("sku")["ean_code"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="snapshot_eans")
)

# Only compare SKUs that exist in BOTH datasets
df_sku_ean_comparison = customer_mapping.merge(
    snap_mapping,
    on="sku",
    how="inner",
    validate="one_to_one"
)

# Identify SKUs whose sets of EANs differ
df_sku_ean_differences = df_sku_ean_comparison[
    df_sku_ean_comparison.apply(
        lambda row: set(row["customer_eans"])
        != set(row["snapshot_eans"]),
        axis=1
    )
].copy()


# ============================================================
# 5. Classify differences
# ============================================================

def classify_ean_difference(row):
    customer_eans = set(row["customer_eans"])
    snapshot_eans = set(row["snapshot_eans"])

    if len(customer_eans) > 1 or len(snapshot_eans) > 1:
        return "Multiple EANs in at least one dataset"

    return "One EAN per dataset, but different EANs"


df_sku_ean_differences["difference_type"] = (
    df_sku_ean_differences.apply(
        classify_ean_difference,
        axis=1
    )
)


# ============================================================
# 6. Display results
# ============================================================

print("\n=== CROSS-DATASET SKU-EAN DIFFERENCES ===")

print(
    "SKUs present in both datasets:",
    len(df_sku_ean_comparison)
)

print(
    "SKUs with different EAN sets:",
    len(df_sku_ean_differences)
)

print("\nBreakdown:")
print(df_sku_ean_differences["difference_type"].value_counts())

display(
    df_sku_ean_differences.sort_values("sku")
)

=== MULTIPLE EANS PER SKU ===
Customer: 0 SKUs
Snapshots: 11 SKUs


,sku,customer_eans


,sku,snapshot_eans
0,24542b78-1b79-4863-bdce-4da9eafcc638,"[67000012860, 067000012860]"
1,27feb930-2213-4bd2-9962-13662ad94a25,"[79581184010, 079581184010]"
2,3e6398fa-b66d-4eb6-aeb1-2932aae017d8,"[24000011859, 024000011859]"
3,5351c7a9-96e2-498a-9efa-d4d804cac62f,"[24000040095, 024000040095]"
4,cd542a42-b55c-4ce5-8006-75a730247bb2,"[24000197409, 024000197409]"
5,d6aeb5ca-43b3-4c75-8f24-7cfcfe768373,"[24000025528, 024000025528]"
6,de6cee29-8de3-4964-bbc6-daa4251605e1,"[41420078477, 041420078477]"
7,e12c96af-2b2e-4e74-aae0-c20b8f1263e4,"[70177176914, 070177176914]"
8,e58bdae4-0fdc-453f-9e16-f6608bb960d7,"[71570017194, 071570017194]"
9,f69a3f7c-fe78-47e2-871c-4e7217709a02,"[67000012860, 067000012860]"



=== CROSS-DATASET SKU-EAN DIFFERENCES ===
SKUs present in both datasets: 372972
SKUs with different EAN sets: 8

Breakdown:
difference_type
Multiple EANs in at least one dataset      5
One EAN per dataset, but different EANs    3
Name: count, dtype: int64


,sku,customer_eans,snapshot_eans,difference_type
52504,24542b78-1b79-4863-bdce-4da9eafcc638,[067000012860],"[067000012860, 67000012860]",Multiple EANs in at least one dataset
57750,27feb930-2213-4bd2-9962-13662ad94a25,[079581184010],"[079581184010, 79581184010]",Multiple EANs in at least one dataset
120384,5351c7a9-96e2-498a-9efa-d4d804cac62f,[024000040095],"[024000040095, 24000040095]",Multiple EANs in at least one dataset
233071,ZasELirVqScsHzHXz9z3,[3760207115033],[3760207115019],"One EAN per dataset, but different EANs"
269123,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277,[1556794],[H1556794],"One EAN per dataset, but different EANs"
298270,cd542a42-b55c-4ce5-8006-75a730247bb2,[024000197409],"[024000197409, 24000197409]",Multiple EANs in at least one dataset
312018,d6aeb5ca-43b3-4c75-8f24-7cfcfe768373,[024000025528],"[024000025528, 24000025528]",Multiple EANs in at least one dataset
365519,fb84e253-caa6-4e94-b9ad-4a91bf33ad31,[87148111754830],[8714811175483],"One EAN per dataset, but different EANs"


In [5]:
df_single_ean_differences = df_sku_ean_differences[
    df_sku_ean_differences["difference_type"]
    == "One EAN per dataset, but different EANs"
].copy()

display(df_single_ean_differences)

print(
    "SKUs with one EAN in each dataset but different assignments:",
    len(df_single_ean_differences)
)

,sku,customer_eans,snapshot_eans,difference_type
233071,ZasELirVqScsHzHXz9z3,[3760207115033],[3760207115019],"One EAN per dataset, but different EANs"
269123,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277,[1556794],[H1556794],"One EAN per dataset, but different EANs"
365519,fb84e253-caa6-4e94-b9ad-4a91bf33ad31,[87148111754830],[8714811175483],"One EAN per dataset, but different EANs"


SKUs with one EAN in each dataset but different assignments: 3


In [6]:
df_multi_ean_differences = df_sku_ean_differences[
    df_sku_ean_differences["difference_type"]
    == "Multiple EANs in at least one dataset"
].copy()

display(df_multi_ean_differences)

print(
    "SKUs with multiple EANs and different mappings:",
    len(df_multi_ean_differences)
)

df_list_snap_ean_investigate = df_multi_ean_differences["snapshot_eans"].tolist()


,sku,customer_eans,snapshot_eans,difference_type
52504,24542b78-1b79-4863-bdce-4da9eafcc638,[067000012860],"[067000012860, 67000012860]",Multiple EANs in at least one dataset
57750,27feb930-2213-4bd2-9962-13662ad94a25,[079581184010],"[079581184010, 79581184010]",Multiple EANs in at least one dataset
120384,5351c7a9-96e2-498a-9efa-d4d804cac62f,[024000040095],"[024000040095, 24000040095]",Multiple EANs in at least one dataset
298270,cd542a42-b55c-4ce5-8006-75a730247bb2,[024000197409],"[024000197409, 24000197409]",Multiple EANs in at least one dataset
312018,d6aeb5ca-43b3-4c75-8f24-7cfcfe768373,[024000025528],"[024000025528, 24000025528]",Multiple EANs in at least one dataset


SKUs with multiple EANs and different mappings: 5


In [7]:
df_list_snap_ean_investigate = ['067000012860', '67000012860', '079581184010', '79581184010', '024000040095', '240000040095', '024000197409', '240000197409', '024000025528', '240000025528']
print(f"List of snapshot EANs to investigate:{len(df_list_snap_ean_investigate)}")
df_prod_ref[df_prod_ref["ean_code"].isin(df_list_snap_ean_investigate)].sort_values("ean_code")
print(df_prod_ref[df_prod_ref["ean_code"].isin(df_list_snap_ean_investigate)].sort_values("ean_code").shape)
display(df_prod_ref[df_prod_ref["ean_code"].isin(df_list_snap_ean_investigate)].sort_values("ean_code"))
list_temp =df_prod_ref[df_prod_ref["ean_code"].isin(df_list_snap_ean_investigate)]["ean_code"].to_list()

if len(list_temp) != len(df_list_snap_ean_investigate):
    missing_eans = set(df_list_snap_ean_investigate) - set(list_temp)
    print(f"Missing EANs in product reference: {missing_eans}")

List of snapshot EANs to investigate:10
(7, 6)


,ean_code,product_concept,main_category,bio,unit,unit_quantity
57,024000025528,fruits en conserve,Epicerie,False,kg,0.420
60,024000040095,fruits en conserve,Epicerie,False,kg,0.140
72,024000197409,fruits en conserve,Epicerie,False,kg,0.681
183,067000012860,limonade,Boissons,False,l,0.355
264,079581184010,limonade,Boissons,False,NaN,NaN
52714,67000012860,limonade,NaN,None,NaN,NaN
54575,79581184010,limonade,Boissons,False,l,0.473


Missing EANs in product reference: {'240000040095', '240000025528', '240000197409'}


In [8]:
# Unique EANs in snapshots
snap_eans = df_prod_snap["ean_code"].dropna().unique()

# Unique EANs in reference
ref_eans = df_prod_ref["ean_code"].dropna().unique()

# EANs present in snapshots but absent from reference
ean_snap_not_ref = snap_eans[
    ~pd.Series(snap_eans).isin(ref_eans).to_numpy()
]

print(f"Unique EANs in snapshots: {len(snap_eans)}")
print(f"Unique EANs in reference: {len(ref_eans)}")
print(f"EANs in snapshots but not reference: {len(ean_snap_not_ref)}")

display(pd.DataFrame({"ean_code": ean_snap_not_ref}))

Unique EANs in snapshots: 63997
Unique EANs in reference: 63997
EANs in snapshots but not reference: 0


,ean_code


In [9]:
import pandas as pd

# ============================================================
# 1. Get distinct SKU-EAN pairs
# ============================================================

snap_pairs = (
    df_prod_snap[["sku", "ean_code"]]
    .dropna(subset=["sku", "ean_code"])
    .drop_duplicates()
)

customer_pairs = (
    df_customer[["sku", "ean_code"]]
    .dropna(subset=["sku", "ean_code"])
    .drop_duplicates()
)

# ============================================================
# 2. Build EAN lists for each SKU
# ============================================================

snap_mapping = (
    snap_pairs
    .groupby("sku")["ean_code"]
    .agg(list)
    .reset_index(name="snapshot_eans")
)

customer_mapping = (
    customer_pairs
    .groupby("sku")["ean_code"]
    .agg(list)
    .reset_index(name="customer_eans")
)

# Attach customer EANs to snapshot SKUs
mapping_review = snap_mapping.merge(
    customer_mapping,
    on="sku",
    how="left",
    validate="one_to_one"
)

# ============================================================
# 3. Resolve the SKU-EAN mapping
# ============================================================

def resolve_ean(row):
    snap_eans = row["snapshot_eans"]
    customer_eans = row["customer_eans"]

    # Case 1: One EAN in snapshots
    if len(snap_eans) == 1:
        return pd.Series({
            "resolved_ean": snap_eans[0],
            "status": "Unique snapshot EAN"
        })

    # Case 2: Multiple snapshot EANs
    if not isinstance(customer_eans, list):
        return pd.Series({
            "resolved_ean": None,
            "status": "No customer mapping"
        })

    # Only consider customer EANs that also exist
    # among this SKU's snapshot EANs
    matching_eans = set(snap_eans) & set(customer_eans)

    if len(customer_eans) == 1 and len(matching_eans) == 1:
        return pd.Series({
            "resolved_ean": next(iter(matching_eans)),
            "status": "Resolved using customer EAN"
        })

    return pd.Series({
        "resolved_ean": None,
        "status": "Manual review required"
    })


mapping_review[["resolved_ean", "status"]] = (
    mapping_review.apply(resolve_ean, axis=1)
)

# ============================================================
# 4. Create final mapping
# ============================================================

snapshot_mapping_resolved = (
    mapping_review[
        mapping_review["resolved_ean"].notna()
    ][["sku", "resolved_ean"]]
    .rename(columns={"resolved_ean": "snapshot_ean_code"})
    .reset_index(drop=True)
)

# ============================================================
# 5. Inspect the 11 ambiguous snapshot SKUs
# ============================================================

df_ambiguous_snap = mapping_review[
    mapping_review["snapshot_eans"].apply(len) > 1
].copy()

print("=== SNAPSHOT SKU-EAN MAPPING ===")
print(f"Total snapshot SKUs: {len(mapping_review)}")
print(f"SKUs with multiple snapshot EANs: {len(df_ambiguous_snap)}")
print(f"Successfully resolved mappings: {len(snapshot_mapping_resolved)}")

print("\nResolution status:")
print(mapping_review["status"].value_counts())

display(df_ambiguous_snap)

# Cases that still require review
df_unresolved = mapping_review[
    mapping_review["resolved_ean"].isna()
].copy()

display(df_unresolved)

=== SNAPSHOT SKU-EAN MAPPING ===
Total snapshot SKUs: 1597714
SKUs with multiple snapshot EANs: 11
Successfully resolved mappings: 1597708

Resolution status:
status
Unique snapshot EAN            1597703
No customer mapping                  6
Resolved using customer EAN          5
Name: count, dtype: int64


,sku,snapshot_eans,customer_eans,resolved_ean,status
225806,24542b78-1b79-4863-bdce-4da9eafcc638,"[67000012860, 067000012860]",[067000012860],067000012860,Resolved using customer EAN
248463,27feb930-2213-4bd2-9962-13662ad94a25,"[79581184010, 079581184010]",[079581184010],079581184010,Resolved using customer EAN
388198,3e6398fa-b66d-4eb6-aeb1-2932aae017d8,"[24000011859, 024000011859]",NaN,NaN,No customer mapping
518742,5351c7a9-96e2-498a-9efa-d4d804cac62f,"[24000040095, 024000040095]",[024000040095],024000040095,Resolved using customer EAN
1280885,cd542a42-b55c-4ce5-8006-75a730247bb2,"[24000197409, 024000197409]",[024000197409],024000197409,Resolved using customer EAN
1339020,d6aeb5ca-43b3-4c75-8f24-7cfcfe768373,"[24000025528, 024000025528]",[024000025528],024000025528,Resolved using customer EAN
1387133,de6cee29-8de3-4964-bbc6-daa4251605e1,"[41420078477, 041420078477]",NaN,NaN,No customer mapping
1404148,e12c96af-2b2e-4e74-aae0-c20b8f1263e4,"[70177176914, 070177176914]",NaN,NaN,No customer mapping
1431591,e58bdae4-0fdc-453f-9e16-f6608bb960d7,"[71570017194, 071570017194]",NaN,NaN,No customer mapping
1538033,f69a3f7c-fe78-47e2-871c-4e7217709a02,"[67000012860, 067000012860]",NaN,NaN,No customer mapping


,sku,snapshot_eans,customer_eans,resolved_ean,status
388198,3e6398fa-b66d-4eb6-aeb1-2932aae017d8,"[24000011859, 024000011859]",NaN,NaN,No customer mapping
1387133,de6cee29-8de3-4964-bbc6-daa4251605e1,"[41420078477, 041420078477]",NaN,NaN,No customer mapping
1404148,e12c96af-2b2e-4e74-aae0-c20b8f1263e4,"[70177176914, 070177176914]",NaN,NaN,No customer mapping
1431591,e58bdae4-0fdc-453f-9e16-f6608bb960d7,"[71570017194, 071570017194]",NaN,NaN,No customer mapping
1538033,f69a3f7c-fe78-47e2-871c-4e7217709a02,"[67000012860, 067000012860]",NaN,NaN,No customer mapping
1585361,fe350efe-815f-4e2c-b235-92f0fc8cfa96,"[701177070236, 0701177070236]",NaN,NaN,No customer mapping


In [10]:

# ============================================================
# 6. RESOLVE EANS THAT DIFFER ONLY BY LEADING ZEROS
# ============================================================

def resolve_leading_zero_eans(ean_list):
    """
    Resolve multiple EANs if they differ only by leading zeros.
    Preserve the representation with the most leading zeros.
    """

    if not isinstance(ean_list, list) or len(ean_list) < 2:
        return None

    # Convert EANs to strings
    eans = [str(ean).strip() for ean in ean_list]

    # Check whether all EANs are equivalent without leading zeros
    normalized = {ean.lstrip("0") or "0" for ean in eans}

    if len(normalized) != 1:
        return None

    # Select the EAN with the most leading zeros
    return max(
        eans,
        key=lambda ean: len(ean) - len(ean.lstrip("0"))
    )


# ============================================================
# 7. IDENTIFY RESOLVABLE CASES
# ============================================================

# Only inspect currently unresolved SKUs
unresolved_mask = mapping_review["resolved_ean"].isna()

# Calculate proposed corrections
mapping_review.loc[
    unresolved_mask, "leading_zero_ean"
] = mapping_review.loc[
    unresolved_mask, "snapshot_eans"
].apply(resolve_leading_zero_eans)


# Preview corrections before applying them
df_leading_zero_fixes = mapping_review[
    mapping_review["resolved_ean"].isna()
    & mapping_review["leading_zero_ean"].notna()
][
    ["sku", "snapshot_eans", "leading_zero_ean"]
].copy()

print("=== LEADING ZERO CORRECTIONS ===")
print(f"Resolvable SKUs: {len(df_leading_zero_fixes)}")

display(df_leading_zero_fixes)


# ============================================================
# 8. APPLY CORRECTIONS TO MAPPING REVIEW
# ============================================================

fix_mask = (
    mapping_review["resolved_ean"].isna()
    & mapping_review["leading_zero_ean"].notna()
)

mapping_review.loc[
    fix_mask, "resolved_ean"
] = mapping_review.loc[
    fix_mask, "leading_zero_ean"
]

mapping_review.loc[
    fix_mask, "status"
] = "Resolved using leading-zero normalization"


# ============================================================
# 9. REBUILD FINAL RESOLVED SNAPSHOT MAPPING
# ============================================================

snapshot_mapping_resolved = (
    mapping_review[
        mapping_review["resolved_ean"].notna()
    ][["sku", "resolved_ean"]]
    .rename(columns={"resolved_ean": "snapshot_ean_code"})
    .reset_index(drop=True)
)


# ============================================================
# 10. UPDATE UNRESOLVED CASES
# ============================================================

df_unresolved = mapping_review[
    mapping_review["resolved_ean"].isna()
].copy()


# ============================================================
# 11. VALIDATE AND DISPLAY RESULTS
# ============================================================

assert snapshot_mapping_resolved["sku"].is_unique, (
    "ERROR: Multiple resolved mappings for the same SKU."
)

print("\n=== UPDATED SNAPSHOT MAPPING ===")

print(f"Total snapshot SKUs: {len(mapping_review)}")

print(
    f"Successfully resolved mappings: "
    f"{len(snapshot_mapping_resolved)}"
)

print(f"Remaining unresolved SKUs: {len(df_unresolved)}")

print("\nResolution status:")
print(mapping_review["status"].value_counts())

print("\n=== CORRECTED LEADING-ZERO CASES ===")
display(
    mapping_review.loc[
        fix_mask,
        ["sku", "snapshot_eans", "resolved_ean", "status"]
    ]
)

print("\n=== REMAINING UNRESOLVED CASES ===")
display(df_unresolved)

=== LEADING ZERO CORRECTIONS ===
Resolvable SKUs: 6


,sku,snapshot_eans,leading_zero_ean
388198,3e6398fa-b66d-4eb6-aeb1-2932aae017d8,"[24000011859, 024000011859]",024000011859
1387133,de6cee29-8de3-4964-bbc6-daa4251605e1,"[41420078477, 041420078477]",041420078477
1404148,e12c96af-2b2e-4e74-aae0-c20b8f1263e4,"[70177176914, 070177176914]",070177176914
1431591,e58bdae4-0fdc-453f-9e16-f6608bb960d7,"[71570017194, 071570017194]",071570017194
1538033,f69a3f7c-fe78-47e2-871c-4e7217709a02,"[67000012860, 067000012860]",067000012860
1585361,fe350efe-815f-4e2c-b235-92f0fc8cfa96,"[701177070236, 0701177070236]",0701177070236



=== UPDATED SNAPSHOT MAPPING ===
Total snapshot SKUs: 1597714
Successfully resolved mappings: 1597714
Remaining unresolved SKUs: 0

Resolution status:
status
Unique snapshot EAN                          1597703
Resolved using leading-zero normalization          6
Resolved using customer EAN                        5
Name: count, dtype: int64

=== CORRECTED LEADING-ZERO CASES ===


,sku,snapshot_eans,resolved_ean,status
388198,3e6398fa-b66d-4eb6-aeb1-2932aae017d8,"[24000011859, 024000011859]",024000011859,Resolved using leading-zero normalization
1387133,de6cee29-8de3-4964-bbc6-daa4251605e1,"[41420078477, 041420078477]",041420078477,Resolved using leading-zero normalization
1404148,e12c96af-2b2e-4e74-aae0-c20b8f1263e4,"[70177176914, 070177176914]",070177176914,Resolved using leading-zero normalization
1431591,e58bdae4-0fdc-453f-9e16-f6608bb960d7,"[71570017194, 071570017194]",071570017194,Resolved using leading-zero normalization
1538033,f69a3f7c-fe78-47e2-871c-4e7217709a02,"[67000012860, 067000012860]",067000012860,Resolved using leading-zero normalization
1585361,fe350efe-815f-4e2c-b235-92f0fc8cfa96,"[701177070236, 0701177070236]",0701177070236,Resolved using leading-zero normalization



=== REMAINING UNRESOLVED CASES ===


,sku,snapshot_eans,customer_eans,resolved_ean,status,leading_zero_ean


In [11]:
import pandas as pd

# ============================================================
# 1. PREPARE DISTINCT SKU-EAN PAIRS
# ============================================================

customer_pairs = (
    df_cust_ord[["sku", "ean_code"]]
    .dropna(subset=["sku", "ean_code"])
    .drop_duplicates()
)

# Original snapshot pairs for data quality reporting
snap_pairs_original = (
    df_prod_snap[["sku", "ean_code"]]
    .dropna(subset=["sku", "ean_code"])
    .drop_duplicates()
)

# Resolved snapshot pairs for comparison
snap_pairs_resolved = (
    snapshot_mapping_resolved[["sku", "snapshot_ean_code"]]
    .rename(columns={"snapshot_ean_code": "ean_code"})
    .dropna(subset=["sku", "ean_code"])
    .drop_duplicates()
)

# Verify one EAN per SKU in resolved mapping
assert not snap_pairs_resolved["sku"].duplicated().any(), (
    "Resolved snapshot mapping contains duplicate SKUs."
)


# ============================================================
# 2. FIND MULTIPLE EANS PER SKU IN ORIGINAL DATASETS
# ============================================================

customer_ean_counts = (
    customer_pairs.groupby("sku")["ean_code"].nunique()
)

snap_ean_counts_original = (
    snap_pairs_original.groupby("sku")["ean_code"].nunique()
)

multi_ean_skus_customer = customer_ean_counts[
    customer_ean_counts > 1
].index

multi_ean_skus_snap = snap_ean_counts_original[
    snap_ean_counts_original > 1
].index


df_multi_ean_customer = (
    customer_pairs[
        customer_pairs["sku"].isin(multi_ean_skus_customer)
    ]
    .groupby("sku")["ean_code"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="customer_eans")
)

df_multi_ean_snap = (
    snap_pairs_original[
        snap_pairs_original["sku"].isin(multi_ean_skus_snap)
    ]
    .groupby("sku")["ean_code"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="original_snapshot_eans")
)


print("=== ORIGINAL MULTI-EAN ANALYSIS ===")
print(f"Customer SKUs with multiple EANs: {len(df_multi_ean_customer)}")
print(f"Snapshot SKUs with multiple EANs: {len(df_multi_ean_snap)}")

print("\nCustomer multi-EAN SKUs:")
display(df_multi_ean_customer)

print("\nOriginal snapshot multi-EAN SKUs:")
display(df_multi_ean_snap)


# ============================================================
# 3. CHECK RESOLUTION OF ORIGINAL SNAPSHOT AMBIGUITIES
# ============================================================

df_snapshot_resolution = (
    mapping_review[
        mapping_review["snapshot_eans"].apply(len) > 1
    ][
        ["sku", "snapshot_eans", "customer_eans",
         "resolved_ean", "status"]
    ]
    .copy()
)

print("\n=== SNAPSHOT AMBIGUITY RESOLUTION ===")

print(f"Originally ambiguous SKUs: {len(df_snapshot_resolution)}")

print(
    "Successfully resolved:",
    df_snapshot_resolution["resolved_ean"].notna().sum()
)

print(
    "Still unresolved:",
    df_snapshot_resolution["resolved_ean"].isna().sum()
)

display(df_snapshot_resolution)


# ============================================================
# 4. BUILD CUSTOMER AND RESOLVED SNAPSHOT MAPPINGS
# ============================================================

customer_mapping = (
    customer_pairs
    .groupby("sku")["ean_code"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="customer_eans")
)

snap_mapping = (
    snap_pairs_resolved
    .groupby("sku")["ean_code"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="snapshot_eans")
)


# ============================================================
# 5. COMPARE CUSTOMER VS RESOLVED SNAPSHOT EAN SETS
# ============================================================

# Compare only SKUs present in both datasets
df_sku_ean_comparison = customer_mapping.merge(
    snap_mapping,
    on="sku",
    how="inner",
    validate="one_to_one"
)

# Add resolution status
df_sku_ean_comparison = df_sku_ean_comparison.merge(
    mapping_review[["sku", "status"]],
    on="sku",
    how="left",
    validate="one_to_one"
)

# Check whether EAN sets match
df_sku_ean_comparison["ean_sets_match"] = (
    df_sku_ean_comparison.apply(
        lambda row: set(row["customer_eans"])
        == set(row["snapshot_eans"]),
        axis=1
    )
)

# Extract mismatches
df_sku_ean_differences = (
    df_sku_ean_comparison[
        ~df_sku_ean_comparison["ean_sets_match"]
    ]
    .copy()
)


# ============================================================
# 6. CLASSIFY MISMATCHES
# ============================================================

def classify_ean_difference(row):

    customer_eans = set(row["customer_eans"])
    snapshot_eans = set(row["snapshot_eans"])

    if len(customer_eans) > 1:
        return "Multiple customer EANs vs resolved snapshot EAN"

    if len(snapshot_eans) > 1:
        return "Multiple snapshot EANs"

    return "One EAN per dataset, but different EANs"


df_sku_ean_differences["difference_type"] = (
    df_sku_ean_differences.apply(
        classify_ean_difference,
        axis=1
    )
)


# ============================================================
# 7. DISPLAY ALL CROSS-DATASET DIFFERENCES
# ============================================================

print("\n=== CUSTOMER VS RESOLVED SNAPSHOT COMPARISON ===")

print(
    "SKUs present in both datasets:",
    len(df_sku_ean_comparison)
)

print(
    "SKUs with matching EAN sets:",
    df_sku_ean_comparison["ean_sets_match"].sum()
)

print(
    "SKUs with different EAN sets:",
    len(df_sku_ean_differences)
)

print("\nDifference types:")
print(df_sku_ean_differences["difference_type"].value_counts())

display(
    df_sku_ean_differences[
        [
            "sku",
            "customer_eans",
            "snapshot_eans",
            "difference_type",
            "status"
        ]
    ].sort_values("sku")
)


# ============================================================
# 8. SEPARATE SINGLE-EAN MISMATCHES
# ============================================================

df_single_ean_differences = (
    df_sku_ean_differences[
        df_sku_ean_differences["difference_type"]
        == "One EAN per dataset, but different EANs"
    ]
    .copy()
)

print("\n=== SINGLE-EAN MISMATCHES ===")

print(
    "SKUs with one EAN in customer and one different EAN "
    "in resolved snapshots:",
    len(df_single_ean_differences)
)

display(df_single_ean_differences)


# ============================================================
# 9. SEPARATE MULTI-EAN CUSTOMER MISMATCHES
# ============================================================

df_multi_ean_differences = (
    df_sku_ean_differences[
        df_sku_ean_differences["difference_type"]
        == "Multiple customer EANs vs resolved snapshot EAN"
    ]
    .copy()
)

print("\n=== MULTI-EAN CUSTOMER MISMATCHES ===")

print(
    "SKUs with multiple customer EANs and different "
    "resolved snapshot mappings:",
    len(df_multi_ean_differences)
)

display(df_multi_ean_differences)


# ============================================================
# 10. SUMMARY
# ============================================================

print("\n========== FINAL SUMMARY ==========")

print(f"Original multi-EAN customer SKUs: {len(df_multi_ean_customer)}")
print(f"Original multi-EAN snapshot SKUs: {len(df_multi_ean_snap)}")

print(
    "Resolved snapshot SKUs:",
    len(snap_pairs_resolved)
)

print(
    "Unresolved snapshot SKUs:",
    mapping_review["resolved_ean"].isna().sum()
)

print(
    "Customer vs resolved snapshot mismatches:",
    len(df_sku_ean_differences)
)

print(
    "Single-EAN mismatches:",
    len(df_single_ean_differences)
)

print(
    "Multi-EAN customer mismatches:",
    len(df_multi_ean_differences)
)

=== ORIGINAL MULTI-EAN ANALYSIS ===
Customer SKUs with multiple EANs: 0
Snapshot SKUs with multiple EANs: 11

Customer multi-EAN SKUs:


,sku,customer_eans



Original snapshot multi-EAN SKUs:


,sku,original_snapshot_eans
0,24542b78-1b79-4863-bdce-4da9eafcc638,"[067000012860, 67000012860]"
1,27feb930-2213-4bd2-9962-13662ad94a25,"[079581184010, 79581184010]"
2,3e6398fa-b66d-4eb6-aeb1-2932aae017d8,"[024000011859, 24000011859]"
3,5351c7a9-96e2-498a-9efa-d4d804cac62f,"[024000040095, 24000040095]"
4,cd542a42-b55c-4ce5-8006-75a730247bb2,"[024000197409, 24000197409]"
5,d6aeb5ca-43b3-4c75-8f24-7cfcfe768373,"[024000025528, 24000025528]"
6,de6cee29-8de3-4964-bbc6-daa4251605e1,"[041420078477, 41420078477]"
7,e12c96af-2b2e-4e74-aae0-c20b8f1263e4,"[070177176914, 70177176914]"
8,e58bdae4-0fdc-453f-9e16-f6608bb960d7,"[071570017194, 71570017194]"
9,f69a3f7c-fe78-47e2-871c-4e7217709a02,"[067000012860, 67000012860]"



=== SNAPSHOT AMBIGUITY RESOLUTION ===
Originally ambiguous SKUs: 11
Successfully resolved: 11
Still unresolved: 0


,sku,snapshot_eans,customer_eans,resolved_ean,status
225806,24542b78-1b79-4863-bdce-4da9eafcc638,"[67000012860, 067000012860]",[067000012860],067000012860,Resolved using customer EAN
248463,27feb930-2213-4bd2-9962-13662ad94a25,"[79581184010, 079581184010]",[079581184010],079581184010,Resolved using customer EAN
388198,3e6398fa-b66d-4eb6-aeb1-2932aae017d8,"[24000011859, 024000011859]",NaN,024000011859,Resolved using leading-zero normalization
518742,5351c7a9-96e2-498a-9efa-d4d804cac62f,"[24000040095, 024000040095]",[024000040095],024000040095,Resolved using customer EAN
1280885,cd542a42-b55c-4ce5-8006-75a730247bb2,"[24000197409, 024000197409]",[024000197409],024000197409,Resolved using customer EAN
1339020,d6aeb5ca-43b3-4c75-8f24-7cfcfe768373,"[24000025528, 024000025528]",[024000025528],024000025528,Resolved using customer EAN
1387133,de6cee29-8de3-4964-bbc6-daa4251605e1,"[41420078477, 041420078477]",NaN,041420078477,Resolved using leading-zero normalization
1404148,e12c96af-2b2e-4e74-aae0-c20b8f1263e4,"[70177176914, 070177176914]",NaN,070177176914,Resolved using leading-zero normalization
1431591,e58bdae4-0fdc-453f-9e16-f6608bb960d7,"[71570017194, 071570017194]",NaN,071570017194,Resolved using leading-zero normalization
1538033,f69a3f7c-fe78-47e2-871c-4e7217709a02,"[67000012860, 067000012860]",NaN,067000012860,Resolved using leading-zero normalization


... (3 outputs left out)

,sku,customer_eans,snapshot_eans,status,ean_sets_match,difference_type
233071,ZasELirVqScsHzHXz9z3,[3760207115033],[3760207115019],Unique snapshot EAN,False,"One EAN per dataset, but different EANs"
269123,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277,[1556794],[H1556794],Unique snapshot EAN,False,"One EAN per dataset, but different EANs"
365519,fb84e253-caa6-4e94-b9ad-4a91bf33ad31,[87148111754830],[8714811175483],Unique snapshot EAN,False,"One EAN per dataset, but different EANs"



=== MULTI-EAN CUSTOMER MISMATCHES ===
SKUs with multiple customer EANs and different resolved snapshot mappings: 0


,sku,customer_eans,snapshot_eans,status,ean_sets_match,difference_type



========== FINAL SUMMARY ==========
Original multi-EAN customer SKUs: 0
Original multi-EAN snapshot SKUs: 11
Resolved snapshot SKUs: 1597714
Unresolved snapshot SKUs: 0
Customer vs resolved snapshot mismatches: 3
Single-EAN mismatches: 3
Multi-EAN customer mismatches: 0


In [12]:
# ============================================================
# 1. Get SKUs with one EAN per dataset, but different EANs
# ============================================================

df_single_ean_differences = df_sku_ean_differences[
    df_sku_ean_differences["difference_type"]
    == "One EAN per dataset, but different EANs"
].copy()


# ============================================================
# 2. Get pickup EAN codes for these SKUs
# ============================================================

pickup_ean_mapping = (
    df_pickup[
        df_pickup["sku"].isin(df_single_ean_differences["sku"])
    ][["sku", "ean_code"]]
    .dropna(subset=["sku", "ean_code"])
    .drop_duplicates()
    .groupby("sku")["ean_code"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="pickup_eans")
)


# ============================================================
# 3. Merge pickup EANs with customer-snapshot differences
# ============================================================

df_single_ean_with_pickup = (
    df_single_ean_differences
    .merge(
        pickup_ean_mapping,
        on="sku",
        how="left",
        validate="one_to_one"
    )
)


# ============================================================
# 4. Check which dataset pickup agrees with
# ============================================================

def compare_pickup_eans(row):

    if not isinstance(row["pickup_eans"], list):
        return "SKU not found in pickup"

    pickup = set(row["pickup_eans"])
    customer = set(row["customer_eans"])
    snapshot = set(row["snapshot_eans"])

    if pickup == customer:
        return "Pickup matches customer"

    if pickup == snapshot:
        return "Pickup matches snapshot"

    if pickup & customer and pickup & snapshot:
        return "Pickup contains EANs from both"

    if pickup & customer:
        return "Pickup partially matches customer"

    if pickup & snapshot:
        return "Pickup partially matches snapshot"

    return "Pickup matches neither"


df_single_ean_with_pickup["pickup_comparison"] = (
    df_single_ean_with_pickup.apply(
        compare_pickup_eans,
        axis=1
    )
)


# ============================================================
# 5. Display results
# ============================================================

print("=== CUSTOMER VS SNAPSHOT VS PICKUP ===")

print(
    "SKUs with different customer and snapshot EANs:",
    len(df_single_ean_with_pickup)
)

print("\nPickup comparison:")
print(
    df_single_ean_with_pickup["pickup_comparison"].value_counts()
)

display(
    df_single_ean_with_pickup[
        [
            "sku",
            "customer_eans",
            "snapshot_eans",
            "pickup_eans",
            "pickup_comparison",
            "status"
        ]
    ].sort_values("sku")
)

=== CUSTOMER VS SNAPSHOT VS PICKUP ===
SKUs with different customer and snapshot EANs: 3

Pickup comparison:
pickup_comparison
Pickup matches customer    3
Name: count, dtype: int64


,sku,customer_eans,snapshot_eans,pickup_eans,pickup_comparison,status
0,ZasELirVqScsHzHXz9z3,[3760207115033],[3760207115019],[3760207115033],Pickup matches customer,Unique snapshot EAN
1,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277,[1556794],[H1556794],[1556794],Pickup matches customer,Unique snapshot EAN
2,fb84e253-caa6-4e94-b9ad-4a91bf33ad31,[87148111754830],[8714811175483],[87148111754830],Pickup matches customer,Unique snapshot EAN


In [13]:
# ============================================================
# 1. IDENTIFY VERIFIED SKU-EAN CORRECTIONS
# ============================================================

# Keep only cases where pickup matches customer exactly
df_verified_corrections = (
    df_single_ean_with_pickup[
        df_single_ean_with_pickup["pickup_comparison"]
        == "Pickup matches customer"
    ].copy()
)

# Additional safety checks: one EAN in each dataset
df_verified_corrections = df_verified_corrections[
    df_verified_corrections["customer_eans"].apply(
        lambda x: isinstance(x, list) and len(x) == 1
    )
    &
    df_verified_corrections["pickup_eans"].apply(
        lambda x: isinstance(x, list) and len(x) == 1
    )
    &
    df_verified_corrections["snapshot_eans"].apply(
        lambda x: isinstance(x, list) and len(x) == 1
    )
].copy()

# Extract scalar EAN values
df_verified_corrections["old_snapshot_ean"] = (
    df_verified_corrections["snapshot_eans"].str[0]
)

df_verified_corrections["corrected_ean"] = (
    df_verified_corrections["customer_eans"].str[0]
)

df_verified_corrections = df_verified_corrections[
    df_verified_corrections["old_snapshot_ean"]
    != df_verified_corrections["corrected_ean"]
].copy()

assert df_verified_corrections["sku"].is_unique, (
    "Duplicate SKUs found in correction candidates."
)

print("=== VERIFIED SNAPSHOT MAPPING CORRECTIONS ===")
print(f"SKUs to correct: {len(df_verified_corrections)}")

display(
    df_verified_corrections[
        ["sku", "old_snapshot_ean", "corrected_ean",
         "pickup_eans", "status"]
    ]
)


# ============================================================
# 2. BACK UP EXISTING MAPPING
# ============================================================

snapshot_mapping_before_correction = (
    snapshot_mapping_resolved.copy()
)

mapping_review_before_correction = (
    mapping_review.copy()
)


# ============================================================
# 3. APPLY CORRECTIONS TO RESOLVED SNAPSHOT MAPPING
# ============================================================

correction_dict = dict(
    zip(
        df_verified_corrections["sku"],
        df_verified_corrections["corrected_ean"]
    )
)

# Check existing values match the expected old EAN
current_mapping = snapshot_mapping_resolved.set_index(
    "sku"
)["snapshot_ean_code"]

for _, row in df_verified_corrections.iterrows():

    sku = row["sku"]
    old_ean = row["old_snapshot_ean"]

    assert sku in current_mapping.index, (
        f"SKU {sku} missing from resolved mapping."
    )

    assert current_mapping.loc[sku] == old_ean, (
        f"Unexpected EAN for SKU {sku}."
    )

# Apply the changes
snapshot_mapping_resolved["snapshot_ean_code"] = (
    snapshot_mapping_resolved["sku"]
    .map(correction_dict)
    .fillna(snapshot_mapping_resolved["snapshot_ean_code"])
)


# ============================================================
# 4. UPDATE MAPPING REVIEW
# ============================================================

correction_mask = mapping_review["sku"].isin(
    correction_dict.keys()
)

mapping_review.loc[
    correction_mask, "resolved_ean"
] = mapping_review.loc[
    correction_mask, "sku"
].map(correction_dict)

mapping_review.loc[
    correction_mask, "status"
] = "Corrected using matching customer and pickup EAN"


# ============================================================
# 5. VALIDATE FINAL MAPPING
# ============================================================

assert snapshot_mapping_resolved["sku"].is_unique, (
    "Final mapping contains duplicate SKUs."
)

assert snapshot_mapping_resolved[
    "snapshot_ean_code"
].notna().all(), (
    "Final mapping contains null EANs."
)

# Verify corrections were applied
updated_mapping = snapshot_mapping_resolved.set_index(
    "sku"
)["snapshot_ean_code"]

for sku, corrected_ean in correction_dict.items():
    assert updated_mapping.loc[sku] == corrected_ean, (
        f"Correction failed for SKU {sku}."
    )


# ============================================================
# 6. DISPLAY FINAL RESULTS
# ============================================================

print("\n=== SNAPSHOT MAPPING UPDATED ===")

print(f"SKU mappings corrected: {len(correction_dict)}")

print(
    "Total resolved SKU mappings:",
    len(snapshot_mapping_resolved)
)

print("\nUpdated resolution statuses:")
print(mapping_review["status"].value_counts())

display(
    snapshot_mapping_resolved[
        snapshot_mapping_resolved["sku"].isin(
            correction_dict.keys()
        )
    ].sort_values("sku")
)

=== VERIFIED SNAPSHOT MAPPING CORRECTIONS ===
SKUs to correct: 3


,sku,old_snapshot_ean,corrected_ean,pickup_eans,status
0,ZasELirVqScsHzHXz9z3,3760207115019,3760207115033,[3760207115033],Unique snapshot EAN
1,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277,H1556794,1556794,[1556794],Unique snapshot EAN
2,fb84e253-caa6-4e94-b9ad-4a91bf33ad31,8714811175483,87148111754830,[87148111754830],Unique snapshot EAN



=== SNAPSHOT MAPPING UPDATED ===
SKU mappings corrected: 3
Total resolved SKU mappings: 1597714

Updated resolution statuses:
status
Unique snapshot EAN                                 1597700
Resolved using leading-zero normalization                 6
Resolved using customer EAN                               5
Corrected using matching customer and pickup EAN          3
Name: count, dtype: int64


,sku,snapshot_ean_code
998336,ZasELirVqScsHzHXz9z3,3760207115033
1154276,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277,1556794
1568550,fb84e253-caa6-4e94-b9ad-4a91bf33ad31,87148111754830


In [15]:
import pandas as pd

# ============================================================
# 1. BACK UP ORIGINAL PICKUP DATA
# ============================================================

df_pickup_original = df_pickup.copy(deep=True)

# Work on a copy
df_pickup_corrected = df_pickup.copy(deep=True)

# Preserve EAN identifiers as strings
for df in [df_pickup_original, df_pickup_corrected]:
    df["ean_code"] = df["ean_code"].astype("string")

# Prepare corrected snapshot mapping
resolved_mapping = snapshot_mapping_resolved[
    ["sku", "snapshot_ean_code"]
].copy()

resolved_mapping["snapshot_ean_code"] = (
    resolved_mapping["snapshot_ean_code"].astype("string")
)

# Validate uniqueness
assert resolved_mapping["sku"].is_unique, (
    "ERROR: Resolved mapping contains duplicate SKUs."
)

assert resolved_mapping["snapshot_ean_code"].notna().all(), (
    "ERROR: Resolved mapping contains null EANs."
)


# ============================================================
# 2. IDENTIFY AMBIGUOUS SKUs IN PICKUP
# ============================================================

# Count distinct non-null EANs per SKU
pickup_ean_counts = (
    df_pickup_original
    .dropna(subset=["sku"])
    .groupby("sku")["ean_code"]
    .nunique()
)

# SKUs with more than one distinct EAN
ambiguous_pickup_skus = pickup_ean_counts[
    pickup_ean_counts > 1
].index

# All original records involving ambiguous SKUs
df_ambiguous_pickup_records = (
    df_pickup_original[
        df_pickup_original["sku"].isin(ambiguous_pickup_skus)
    ].copy()
)

# Distinct SKU-EAN pairs
df_ambiguous_pickup_pairs = (
    df_ambiguous_pickup_records[
        ["sku", "ean_code"]
    ]
    .dropna(subset=["ean_code"])
    .drop_duplicates()
)

print("=== ORIGINAL PICKUP AMBIGUITIES ===")

print(
    "SKUs with multiple EANs:",
    len(ambiguous_pickup_skus)
)

print(
    "Distinct SKU-EAN pairs:",
    len(df_ambiguous_pickup_pairs)
)

print(
    "Pickup records involving ambiguous SKUs:",
    len(df_ambiguous_pickup_records)
)


# ============================================================
# 3. BUILD SKU-LEVEL AMBIGUITY REPORT
# ============================================================

pickup_ambiguity_summary = (
    df_ambiguous_pickup_records
    .groupby("sku")
    .agg(
        total_records=("sku", "size"),
        unique_eans=("ean_code", "nunique"),
        pickup_eans=(
            "ean_code",
            lambda x: sorted(x.dropna().unique().tolist())
        )
    )
    .reset_index()
)

# Add resolved EAN for each SKU
pickup_ambiguity_summary = (
    pickup_ambiguity_summary.merge(
        resolved_mapping,
        on="sku",
        how="left",
        validate="one_to_one"
    )
)

# Add information from mapping_review
pickup_ambiguity_summary = (
    pickup_ambiguity_summary.merge(
        mapping_review[
            ["sku", "status"]
        ].rename(
            columns={"status": "snapshot_resolution_status"}
        ),
        on="sku",
        how="left",
        validate="one_to_one"
    )
)


# ============================================================
# 4. CLASSIFY AMBIGUOUS PICKUP SKUs
# ============================================================

def classify_pickup_ambiguity(row):

    if pd.notna(row["snapshot_ean_code"]):
        return "Correctable using resolved snapshot mapping"

    if pd.isna(row["snapshot_resolution_status"]):
        return "SKU missing from snapshots"

    return "Snapshot SKU mapping unresolved"


pickup_ambiguity_summary["correction_status"] = (
    pickup_ambiguity_summary.apply(
        classify_pickup_ambiguity,
        axis=1
    )
)


# ============================================================
# 5. CREATE CORRECTION DICTIONARY
# ============================================================

correctable_skus = pickup_ambiguity_summary[
    pickup_ambiguity_summary["correction_status"]
    == "Correctable using resolved snapshot mapping"
].copy()

correction_dict = dict(
    zip(
        correctable_skus["sku"],
        correctable_skus["snapshot_ean_code"]
    )
)


# ============================================================
# 6. IDENTIFY INDIVIDUAL RECORDS REQUIRING CORRECTION
# ============================================================

# Attach expected EAN to pickup records
df_pickup_review = df_pickup_original.merge(
    resolved_mapping,
    on="sku",
    how="left",
    validate="many_to_one"
)

# Only correct ambiguous SKUs with a resolved mapping
correction_mask = (
    df_pickup_review["sku"].isin(correction_dict.keys())
    &
    df_pickup_review["ean_code"].notna()
    &
    df_pickup_review["snapshot_ean_code"].notna()
    &
    (
        df_pickup_review["ean_code"]
        != df_pickup_review["snapshot_ean_code"]
    )
)

# Treat nullable Boolean values safely
correction_mask = correction_mask.fillna(False)

# Audit trail of changed records
df_pickup_corrections = (
    df_pickup_review.loc[correction_mask]
    .copy()
)

df_pickup_corrections = df_pickup_corrections.rename(
    columns={
        "ean_code": "old_ean",
        "snapshot_ean_code": "corrected_ean"
    }
)

# Include original row index for traceability
df_pickup_corrections["original_index"] = (
    df_pickup_corrections.index
)


# ============================================================
# 7. APPLY CORRECTIONS
# ============================================================

# Update only the rows identified by the mask
df_pickup_corrected.loc[
    correction_mask, "ean_code"
] = df_pickup_review.loc[
    correction_mask, "snapshot_ean_code"
].to_numpy()


# ============================================================
# 8. COUNT DISTINCT PAIRS CORRECTED
# ============================================================

df_corrected_pairs = (
    df_pickup_corrections[
        ["sku", "old_ean", "corrected_ean"]
    ]
    .drop_duplicates()
)

# Count number of changed rows per SKU
corrected_records_per_sku = (
    df_pickup_corrections
    .groupby("sku")
    .size()
    .reset_index(name="corrected_records")
)

# Count distinct incorrect SKU-EAN pairs per SKU
corrected_pairs_per_sku = (
    df_corrected_pairs
    .groupby("sku")
    .size()
    .reset_index(name="corrected_pairs")
)

pickup_ambiguity_summary = (
    pickup_ambiguity_summary.merge(
        corrected_records_per_sku,
        on="sku",
        how="left"
    )
    .merge(
        corrected_pairs_per_sku,
        on="sku",
        how="left"
    )
)

pickup_ambiguity_summary[
    ["corrected_records", "corrected_pairs"]
] = (
    pickup_ambiguity_summary[
        ["corrected_records", "corrected_pairs"]
    ]
    .fillna(0)
    .astype(int)
)


# ============================================================
# 9. IDENTIFY UNRESOLVED AMBIGUITIES
# ============================================================

df_unresolved_pickup = (
    pickup_ambiguity_summary[
        pickup_ambiguity_summary["correction_status"]
        != "Correctable using resolved snapshot mapping"
    ]
    .copy()
)

# Specifically: ambiguous pickup SKUs missing from snapshots
df_pickup_missing_snap = (
    df_unresolved_pickup[
        df_unresolved_pickup["correction_status"]
        == "SKU missing from snapshots"
    ]
    .copy()
)

# Specifically: ambiguous pickup SKUs with unresolved snapshot mapping
df_pickup_unresolved_snap = (
    df_unresolved_pickup[
        df_unresolved_pickup["correction_status"]
        == "Snapshot SKU mapping unresolved"
    ]
    .copy()
)


# ============================================================
# 10. VALIDATE CORRECTIONS
# ============================================================

# Original dataframe should not change
assert df_pickup_original.equals(
    df_pickup.astype(
        {"ean_code": "string"}
    )
), "Original pickup data was unexpectedly modified."

# Number of changed records should match audit
actual_changes = (
    df_pickup_original["ean_code"]
    .ne(df_pickup_corrected["ean_code"])
    .fillna(False)
    .sum()
)

assert actual_changes == len(df_pickup_corrections), (
    "ERROR: Changed row count does not match audit trail."
)

# Check remaining multi-EAN SKUs
remaining_ean_counts = (
    df_pickup_corrected
    .groupby("sku")["ean_code"]
    .nunique()
)

remaining_ambiguous_skus = remaining_ean_counts[
    remaining_ean_counts > 1
].index

df_remaining_ambiguous = (
    df_pickup_corrected[
        df_pickup_corrected["sku"].isin(
            remaining_ambiguous_skus
        )
    ]
    .groupby("sku")["ean_code"]
    .agg(lambda x: sorted(x.dropna().unique().tolist()))
    .reset_index(name="remaining_eans")
)


# ============================================================
# 11. PRINT COMPLETE SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("PICKUP SKU-EAN DATA QUALITY CORRECTION REPORT")
print("=" * 60)

print("\n--- ORIGINAL AMBIGUITIES ---")

print(
    f"Ambiguous pickup SKUs: "
    f"{len(ambiguous_pickup_skus):,}"
)

print(
    f"Original distinct SKU-EAN pairs among ambiguous SKUs: "
    f"{len(df_ambiguous_pickup_pairs):,}"
)

print(
    f"Original records involving ambiguous SKUs: "
    f"{len(df_ambiguous_pickup_records):,}"
)

print("\n--- CORRECTIONS ---")

print(
    f"SKUs eligible for correction: "
    f"{len(correctable_skus):,}"
)

print(
    f"Distinct SKU-EAN pairs corrected: "
    f"{len(df_corrected_pairs):,}"
)

print(
    f"Individual pickup records corrected: "
    f"{len(df_pickup_corrections):,}"
)

print("\n--- UNRESOLVED CASES ---")

print(
    f"Ambiguous SKUs not corrected: "
    f"{len(df_unresolved_pickup):,}"
)

print(
    f"Ambiguous SKUs missing from snapshots: "
    f"{len(df_pickup_missing_snap):,}"
)

print(
    f"Ambiguous SKUs with unresolved snapshot mapping: "
    f"{len(df_pickup_unresolved_snap):,}"
)

print(
    f"SKUs still associated with multiple EANs after correction: "
    f"{len(remaining_ambiguous_skus):,}"
)

print("\n--- REASONS ---")

print(
    pickup_ambiguity_summary[
        "correction_status"
    ].value_counts()
)


# ============================================================
# 12. DISPLAY REPORT TABLES
# ============================================================

print("\n=== ALL AMBIGUOUS PICKUP SKUs ===")
display(
    pickup_ambiguity_summary.sort_values(
        ["correction_status", "total_records"],
        ascending=[True, False]
    )
)

print("\n=== CORRECTED SKU-EAN PAIRS ===")
display(
    df_corrected_pairs.sort_values("sku")
)

print("\n=== INDIVIDUAL CORRECTED PICKUP RECORDS ===")
display(df_pickup_corrections)

print("\n=== UNRESOLVED PICKUP AMBIGUITIES ===")
display(df_unresolved_pickup)

print("\n=== AMBIGUOUS PICKUP SKUs MISSING FROM SNAPSHOTS ===")
display(df_pickup_missing_snap)

print("\n=== AMBIGUOUS PICKUP SKUs WITH UNRESOLVED SNAPSHOT MAPPING ===")
display(df_pickup_unresolved_snap)

print("\n=== REMAINING MULTI-EAN SKUs AFTER CORRECTION ===")
display(df_remaining_ambiguous)

=== ORIGINAL PICKUP AMBIGUITIES ===
SKUs with multiple EANs: 35
Distinct SKU-EAN pairs: 78
Pickup records involving ambiguous SKUs: 1525

PICKUP SKU-EAN DATA QUALITY CORRECTION REPORT

--- ORIGINAL AMBIGUITIES ---
Ambiguous pickup SKUs: 35
Original distinct SKU-EAN pairs among ambiguous SKUs: 78
Original records involving ambiguous SKUs: 1,525

--- CORRECTIONS ---
SKUs eligible for correction: 35
Distinct SKU-EAN pairs corrected: 43
Individual pickup records corrected: 53

--- UNRESOLVED CASES ---
Ambiguous SKUs not corrected: 0
Ambiguous SKUs missing from snapshots: 0
Ambiguous SKUs with unresolved snapshot mapping: 0
SKUs still associated with multiple EANs after correction: 0

--- REASONS ---
correction_status
Correctable using resolved snapshot mapping    35
Name: count, dtype: int64

=== ALL AMBIGUOUS PICKUP SKUs ===


,sku,total_records,unique_eans,pickup_eans,snapshot_ean_code,snapshot_resolution_status,correction_status,corrected_records,corrected_pairs
6,15d19704-0ecf-4eb7-9e76-7a3245a16d8d,410,2,['5400601061618' '87365368'],87365368,Unique snapshot EAN,Correctable using resolved snapshot mapping,1,1
0,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,162,7,['3336374001754' '3487400006769' '356007142662...,8719200263796,Unique snapshot EAN,Correctable using resolved snapshot mapping,6,6
8,1ef44333-2795-4b55-a514-dea3e6582ae5,154,3,['5410221205453' '5425020166423' '8002200145491'],8002200145491,Unique snapshot EAN,Correctable using resolved snapshot mapping,2,2
26,d6a34af7-bc99-4efc-8e69-a82869b415e3,151,2,['4820078570979' '5413110052367'],4820078570979,Unique snapshot EAN,Correctable using resolved snapshot mapping,1,1
13,40abbe09-c2d4-47ec-9022-819a6c7fea84,93,2,['5410138051051' '8001100064697'],5410138051051,Unique snapshot EAN,Correctable using resolved snapshot mapping,1,1
31,e6c43336-c92c-4588-bfdc-db1184242add,87,3,['3023290047941' '3228170823404' '5400113594291'],3023290047941,Unique snapshot EAN,Correctable using resolved snapshot mapping,2,2
25,c41703dd-e648-4462-a933-3cda0d0000a1,71,2,['3250392615536' '8719500005690'],8719500005690,Unique snapshot EAN,Correctable using resolved snapshot mapping,1,1
14,5a0fc00e-11ee-4adb-9201-3684848e5b85,66,2,['5420036416210' '9998883'],9998883,Unique snapshot EAN,Correctable using resolved snapshot mapping,1,1
23,affd56f3-0a66-4ac1-a07a-82ff0e73c56c,37,2,['5038862004558' '5400119589635'],5400119589635,Unique snapshot EAN,Correctable using resolved snapshot mapping,1,1
9,1fb27241-c5b0-4334-a2ef-b54987640af9,34,2,['5400119568746' '5410228215066'],5410228215066,Unique snapshot EAN,Correctable using resolved snapshot mapping,1,1



=== CORRECTED SKU-EAN PAIRS ===


,sku,old_ean,corrected_ean
334586,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,3487400006769,8719200263796
353293,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,5400119566766,8719200263796
357812,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,4014071692718,8719200263796
362720,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,5410138049256,8719200263796
362721,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,3336374001754,8719200263796
365756,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,3560071426620,8719200263796
167500,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100
341213,0ab97db6-bd06-4ca5-8fd4-c5fe28e9a5a0,5400119537643,5400113700449
363033,104844f8-1bd9-4a67-b4de-132854460dc6,7613287419095,5410146409813
555253,12baf391-2b31-477b-a7f4-855a3e836275,8710398162045,5413883005522



=== INDIVIDUAL CORRECTED PICKUP RECORDS ===


,order_date,pickup_point,sku,old_ean,quantity_ordered,new_price,n_orders,corrected_ean,original_index
167500,2024-03-06,092a6887b4c84db6,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,8,1.10,8,871039852411900100,167500
167889,2024-03-06,0e7ecfc90eb1dd42,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,9,1.10,9,871039852411900100,167889
168161,2024-03-06,142edab1280b2246,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,19,1.10,11,871039852411900100,168161
168558,2024-03-06,1a1c842a643d33ce,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,13,1.10,10,871039852411900100,168558
168786,2024-03-06,2542941672fd21b8,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,11,1.10,8,871039852411900100,168786
169042,2024-03-06,2e815a32c23cc1e2,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,15,1.10,11,871039852411900100,169042
169245,2024-03-06,48db013424cf43ec,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,11,1.10,9,871039852411900100,169245
169432,2024-03-06,78a2806cc9346b6e,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,8,1.10,6,871039852411900100,169432
169626,2024-03-06,be903990255560d6,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,7,1.10,7,871039852411900100,169626
169859,2024-03-06,cbcb62162e461101,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,22,1.10,17,871039852411900100,169859


... (4 outputs left out)


=== AMBIGUOUS PICKUP SKUs WITH UNRESOLVED SNAPSHOT MAPPING ===


,sku,total_records,unique_eans,pickup_eans,snapshot_ean_code,snapshot_resolution_status,correction_status,corrected_records,corrected_pairs



=== REMAINING MULTI-EAN SKUs AFTER CORRECTION ===


,sku,remaining_eans


DataFrame	Purpose
df_pickup_corrected	Full pickup dataset with corrected EANs
df_pickup_original	Original pickup data preserved
df_pickup_corrections	Row-level audit trail showing old and corrected EANs
df_corrected_pairs	Distinct SKU–EAN combinations corrected
pickup_ambiguity_summary	Full SKU-level data quality report
df_unresolved_pickup	Ambiguous SKUs that could not be corrected
df_pickup_missing_snap	Ambiguous pickup SKUs missing from snapshot mapping
df_pickup_unresolved_snap	Ambiguous pickup SKUs whose snapshot mapping is unresolved
df_remaining_ambiguous	SKUs still associated with multiple EANs after correction

In [27]:

import pandas as pd

# ============================================================
# 1. PREPARE DISTINCT SKU-EAN PAIRS
# ============================================================

def prepare_pairs(df, ean_column="ean_code"):
    pairs = (
        df[["sku", ean_column]]
        .rename(columns={ean_column: "ean_code"})
        .copy()
    )

    # Preserve leading zeros in EAN identifiers
    pairs["sku"] = pairs["sku"].astype("string").str.strip()
    pairs["ean_code"] = pairs["ean_code"].astype("string").str.strip()

    pairs["sku"] = pairs["sku"].replace("", pd.NA)
    pairs["ean_code"] = pairs["ean_code"].replace("", pd.NA)

    return (
        pairs
        .dropna(subset=["sku", "ean_code"])
        .drop_duplicates()
        .reset_index(drop=True)
    )


customer_pairs = prepare_pairs(df_cust_ord)
pickup_pairs = prepare_pairs(df_pickup_corrected)

snapshot_pairs = prepare_pairs(
    snapshot_mapping_resolved,
    ean_column="snapshot_ean_code"
)

assert snapshot_pairs["sku"].is_unique, (
    "Resolved snapshot mapping contains multiple EANs per SKU."
)


# ============================================================
# 2. BUILD SKU-EAN MAPPINGS
# ============================================================

def build_sku_mapping(pairs, column_name):

    result = (
        pairs
        .groupby("sku")["ean_code"]
        .agg(lambda x: sorted(set(x.astype(str))))
        .reset_index(name=column_name)
    )

    # Explicitly store Python lists in object columns
    result[column_name] = pd.Series(
        result[column_name].tolist(),
        index=result.index,
        dtype=object
    )

    return result


customer_mapping = build_sku_mapping(
    customer_pairs, "customer_eans"
)

pickup_mapping = build_sku_mapping(
    pickup_pairs, "pickup_eans"
)

snapshot_mapping = build_sku_mapping(
    snapshot_pairs, "snapshot_eans"
)


# ============================================================
# 3. MERGE ALL THREE DATASETS
# ============================================================

df_three_way_comparison = (
    customer_mapping
    .merge(
        pickup_mapping,
        on="sku",
        how="outer",
        validate="one_to_one"
    )
    .merge(
        snapshot_mapping,
        on="sku",
        how="outer",
        validate="one_to_one"
    )
)


# ============================================================
# 4. CLASSIFY SKU-EAN CONSISTENCY
# ============================================================

def classify_three_way(row):

    available = {}

    if isinstance(row["customer_eans"], list):
        available["Customer"] = set(row["customer_eans"])

    if isinstance(row["pickup_eans"], list):
        available["Pickup"] = set(row["pickup_eans"])

    if isinstance(row["snapshot_eans"], list):
        available["Snapshot"] = set(row["snapshot_eans"])

    n_available = len(available)

    # CASE 0: No valid EAN mapping
    if n_available == 0:
        return (
            "No valid EAN mapping available",
            False
        )

    # CASE 1: Only one dataset available
    if n_available == 1:
        return (
            "Only one dataset available",
            False
        )

    # CASE 2: Multiple EANs in any available dataset
    if any(len(eans) > 1 for eans in available.values()):
        return (
            "Multiple EANs in at least one dataset",
            True
        )

    # CASE 3: Exactly two datasets available
    if n_available == 2:

        ean_sets = list(available.values())

        if ean_sets[0] == ean_sets[1]:
            return (
                "Two datasets match; third missing",
                False
            )

        return (
            "Two datasets differ; third missing",
            True
        )

    # CASE 4: All three datasets available
    if n_available == 3:

        customer = available["Customer"]
        pickup = available["Pickup"]
        snapshot = available["Snapshot"]

        if customer == pickup == snapshot:
            return ("All three match", False)

        if pickup == snapshot:
            return ("Customer differs", True)

        if customer == snapshot:
            return ("Pickup differs", True)

        if customer == pickup:
            return ("Snapshot differs", True)

        return ("All three differ", True)

    # Defensive fallback
    return ("Unexpected dataset combination", True)


# Apply classification safely
classification = df_three_way_comparison.apply(
    classify_three_way,
    axis=1
)

# Convert tuples into two separate columns
df_three_way_comparison["consistency_status"] = (
    classification.map(lambda x: x[0])
)

df_three_way_comparison["is_flagged"] = (
    classification.map(lambda x: x[1]).astype(bool)
)


# ============================================================
# 5. ADD DATASET AVAILABILITY INFORMATION
# ============================================================

ean_columns = {
    "Customer": "customer_eans",
    "Pickup": "pickup_eans",
    "Snapshot": "snapshot_eans"
}


def get_available_datasets(row):
    return ", ".join(
        name
        for name, column in ean_columns.items()
        if isinstance(row[column], list)
    )


def get_missing_datasets(row):
    return ", ".join(
        name
        for name, column in ean_columns.items()
        if not isinstance(row[column], list)
    )


df_three_way_comparison["available_datasets"] = (
    df_three_way_comparison.apply(
        get_available_datasets,
        axis=1
    )
)

df_three_way_comparison["missing_datasets"] = (
    df_three_way_comparison.apply(
        get_missing_datasets,
        axis=1
    )
)

# ============================================================
# COUNT AVAILABLE DATASETS SAFELY
# ============================================================

ean_columns = [
    "customer_eans",
    "pickup_eans",
    "snapshot_eans"
]

# Count non-null mappings without row-wise apply
df_three_way_comparison["number_of_sources"] = (
    df_three_way_comparison[ean_columns]
    .notna()
    .sum(axis=1)
    .astype(int)
)


# ============================================================
# 6. EXTRACT REPORT TABLES
# ============================================================

# All flagged inconsistencies
df_three_way_differences = (
    df_three_way_comparison[
        df_three_way_comparison["is_flagged"]
    ]
    .copy()
)

# Flagged cases with all three datasets present
df_actual_ean_conflicts = (
    df_three_way_differences[
        df_three_way_differences["number_of_sources"] == 3
    ]
    .copy()
)

# Flagged cases with exactly two datasets present
df_two_dataset_conflicts = (
    df_three_way_differences[
        df_three_way_differences["number_of_sources"] == 2
    ]
    .copy()
)

# Two datasets match, third missing
df_two_dataset_matches = (
    df_three_way_comparison[
        df_three_way_comparison["consistency_status"]
        == "Two datasets match; third missing"
    ]
    .copy()
)

# Only one dataset available
df_single_dataset_skus = (
    df_three_way_comparison[
        df_three_way_comparison["consistency_status"]
        == "Only one dataset available"
    ]
    .copy()
)

# Remaining multiple-EAN cases
df_remaining_multi_ean = (
    df_three_way_comparison[
        df_three_way_comparison["consistency_status"]
        == "Multiple EANs in at least one dataset"
    ]
    .copy()
)

# Single-EAN conflicts only
single_ean_conflict_types = [
    "Customer differs",
    "Pickup differs",
    "Snapshot differs",
    "All three differ",
    "Two datasets differ; third missing"
]

df_single_ean_conflicts = (
    df_three_way_comparison[
        df_three_way_comparison["consistency_status"].isin(
            single_ean_conflict_types
        )
    ]
    .copy()
)


# ============================================================
# 7. STATUS EXPLANATIONS
# ============================================================

status_explanations = {
    "All three match":
        "Customer, pickup, and snapshot assign the same unique EAN to the SKU.",

    "Customer differs":
        "Pickup and snapshot agree, but customer assigns a different EAN.",

    "Pickup differs":
        "Customer and snapshot agree, but pickup assigns a different EAN.",

    "Snapshot differs":
        "Customer and pickup agree, but snapshot assigns a different EAN.",

    "All three differ":
        "Each dataset assigns a different EAN to the same SKU.",

    "Two datasets match; third missing":
        "The SKU has a valid mapping in two datasets, and both agree. "
        "The third dataset has no valid mapping. Not flagged.",

    "Two datasets differ; third missing":
        "The SKU has a valid mapping in two datasets, but their EANs disagree. "
        "The third dataset has no valid mapping. Flagged.",

    "Only one dataset available":
        "Only one dataset provides a valid EAN mapping for the SKU. "
        "There is insufficient evidence for a cross-dataset comparison.",

    "Multiple EANs in at least one dataset":
        "At least one dataset associates the SKU with multiple distinct EANs. "
        "Flagged for further investigation.",

    "No valid EAN mapping available":
        "None of the datasets provides a valid SKU-EAN mapping.",

    "Unexpected dataset combination":
        "An unexpected configuration was encountered. "
        "Flagged for investigation."
}


# ============================================================
# 8. PRINT COMPLETE CONSISTENCY SUMMARY
# ============================================================

print("=" * 75)
print("THREE-WAY SKU-EAN CONSISTENCY REPORT")
print("=" * 75)

print("\n=== DATASET COVERAGE ===")

print(f"Unique customer SKUs: {len(customer_mapping):,}")
print(f"Unique pickup SKUs: {len(pickup_mapping):,}")
print(f"Unique snapshot SKUs: {len(snapshot_mapping):,}")

print(
    f"Total distinct SKUs across all datasets: "
    f"{len(df_three_way_comparison):,}"
)

print("\n=== CONSISTENCY STATUS BREAKDOWN ===")

status_counts = (
    df_three_way_comparison["consistency_status"]
    .value_counts()
)

for status, count in status_counts.items():
    print(f"\n{status}: {count:,}")
    print(f"  Meaning: {status_explanations[status]}")


print("\n=== OVERALL SUMMARY ===")

print(
    f"Total flagged inconsistencies: "
    f"{len(df_three_way_differences):,}"
)

print(
    f"Flagged cases with all three datasets present: "
    f"{len(df_actual_ean_conflicts):,}"
)

print(
    f"Flagged cases with exactly two datasets present: "
    f"{len(df_two_dataset_conflicts):,}"
)

print(
    f"Two datasets matching, third missing (not flagged): "
    f"{len(df_two_dataset_matches):,}"
)

print(
    f"SKUs present in only one dataset (not flagged): "
    f"{len(df_single_dataset_skus):,}"
)

print(
    f"Remaining multi-EAN SKUs: "
    f"{len(df_remaining_multi_ean):,}"
)

print(
    f"Single-EAN conflicts: "
    f"{len(df_single_ean_conflicts):,}"
)


# ============================================================
# 9. DISPLAY DETAILED REPORTS
# ============================================================

print("\n=== ALL FLAGGED INCONSISTENCIES ===")

display(
    df_three_way_differences.sort_values(
        ["consistency_status", "sku"]
    )
)

print("\n=== FLAGGED: ALL THREE DATASETS PRESENT ===")

display(
    df_actual_ean_conflicts.sort_values(
        ["consistency_status", "sku"]
    )
)

print("\n=== FLAGGED: TWO DATASETS DIFFER, THIRD MISSING ===")

display(
    df_two_dataset_conflicts.sort_values(
        ["consistency_status", "sku"]
    )
)

print("\n=== NOT FLAGGED: TWO DATASETS MATCH, THIRD MISSING ===")

display(
    df_two_dataset_matches.sort_values("sku")
)

print("\n=== NOT FLAGGED: ONLY ONE DATASET AVAILABLE ===")

display(
    df_single_dataset_skus.sort_values("sku")
)

print("\n=== REMAINING MULTI-EAN CASES ===")

display(
    df_remaining_multi_ean.sort_values("sku")
)

print("\n=== SINGLE-EAN CONFLICTS ONLY ===")

display(
    df_single_ean_conflicts.sort_values(
        ["consistency_status", "sku"]
    )
)


# ============================================================
# 10. VALIDATION CHECKS
# ============================================================

# Every SKU must belong to exactly one consistency category
assert status_counts.sum() == len(df_three_way_comparison)

# Flagged + unflagged must equal all SKUs
assert (
    df_three_way_comparison["is_flagged"].sum()
    + (~df_three_way_comparison["is_flagged"]).sum()
    == len(df_three_way_comparison)
)

# Two matching datasets with third missing must never be flagged
assert not df_two_dataset_matches["is_flagged"].any()

# Two conflicting datasets with third missing must be flagged
assert df_two_dataset_conflicts["is_flagged"].all()

print("\nAll consistency validation checks passed.")


THREE-WAY SKU-EAN CONSISTENCY REPORT

=== DATASET COVERAGE ===
Unique customer SKUs: 373,117
Unique pickup SKUs: 1,304,291
Unique snapshot SKUs: 1,597,714
Total distinct SKUs across all datasets: 1,598,368

=== CONSISTENCY STATUS BREAKDOWN ===

Two datasets match; third missing: 930,683
  Meaning: The SKU has a valid mapping in two datasets, and both agree. The third dataset has no valid mapping. Not flagged.

All three match: 372,972
  Meaning: Customer, pickup, and snapshot assign the same unique EAN to the SKU.

Only one dataset available: 294,586
  Meaning: Only one dataset provides a valid EAN mapping for the SKU. There is insufficient evidence for a cross-dataset comparison.

Two datasets differ; third missing: 127
  Meaning: The SKU has a valid mapping in two datasets, but their EANs disagree. The third dataset has no valid mapping. Flagged.

=== OVERALL SUMMARY ===
Total flagged inconsistencies: 127
Flagged cases with all three datasets present: 0
Flagged cases with exactly two

,sku,customer_eans,pickup_eans,snapshot_eans,consistency_status,is_flagged,available_datasets,missing_datasets,number_of_sources
907,0023c4f2-cc0e-4094-8ac0-d647f3c0273d,NaN,[024000017677],[24000017677],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
6481,010a9eb8-8427-4948-b986-dc66a606487b,NaN,[038900033125],[38900033125],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
14744,025b0474-ee07-4d0d-80d2-f0b3f18765fd,NaN,[031200456991],[31200456991],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
19838,032883ef-62e6-42e3-993b-b0fb5eed68bb,NaN,[031200456991],[31200456991],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
35922,05bf4207-2c6e-40b0-9c46-4138b3f2c7ea,NaN,[038900029746],[38900029746],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
...,...,...,...,...,...,...,...,...,...
1557097,f993f5e4-422a-4b38-bc49-8906104478b2,NaN,[070462098358],[70462098358],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
1558263,f9c4e2d1-0afb-411f-9c8f-389286d8c61b,NaN,[024000106654],[24000106654],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
1572985,fc1e656e-4082-405b-b66c-ed68beb00a2a,NaN,[024000197409],[24000197409],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
1577105,fcc9a936-ed1f-406c-8c5e-abe7fcdec6e0,NaN,[038900033125],[38900033125],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2



=== FLAGGED: ALL THREE DATASETS PRESENT ===


,sku,customer_eans,pickup_eans,snapshot_eans,consistency_status,is_flagged,available_datasets,missing_datasets,number_of_sources



=== FLAGGED: TWO DATASETS DIFFER, THIRD MISSING ===


,sku,customer_eans,pickup_eans,snapshot_eans,consistency_status,is_flagged,available_datasets,missing_datasets,number_of_sources
907,0023c4f2-cc0e-4094-8ac0-d647f3c0273d,NaN,[024000017677],[24000017677],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
6481,010a9eb8-8427-4948-b986-dc66a606487b,NaN,[038900033125],[38900033125],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
14744,025b0474-ee07-4d0d-80d2-f0b3f18765fd,NaN,[031200456991],[31200456991],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
19838,032883ef-62e6-42e3-993b-b0fb5eed68bb,NaN,[031200456991],[31200456991],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
35922,05bf4207-2c6e-40b0-9c46-4138b3f2c7ea,NaN,[038900029746],[38900029746],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
...,...,...,...,...,...,...,...,...,...
1557097,f993f5e4-422a-4b38-bc49-8906104478b2,NaN,[070462098358],[70462098358],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
1558263,f9c4e2d1-0afb-411f-9c8f-389286d8c61b,NaN,[024000106654],[24000106654],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
1572985,fc1e656e-4082-405b-b66c-ed68beb00a2a,NaN,[024000197409],[24000197409],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
1577105,fcc9a936-ed1f-406c-8c5e-abe7fcdec6e0,NaN,[038900033125],[38900033125],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2


... (5 outputs left out)

,sku,customer_eans,pickup_eans,snapshot_eans,consistency_status,is_flagged,available_datasets,missing_datasets,number_of_sources



=== SINGLE-EAN CONFLICTS ONLY ===


,sku,customer_eans,pickup_eans,snapshot_eans,consistency_status,is_flagged,available_datasets,missing_datasets,number_of_sources
907,0023c4f2-cc0e-4094-8ac0-d647f3c0273d,NaN,[024000017677],[24000017677],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
6481,010a9eb8-8427-4948-b986-dc66a606487b,NaN,[038900033125],[38900033125],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
14744,025b0474-ee07-4d0d-80d2-f0b3f18765fd,NaN,[031200456991],[31200456991],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
19838,032883ef-62e6-42e3-993b-b0fb5eed68bb,NaN,[031200456991],[31200456991],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
35922,05bf4207-2c6e-40b0-9c46-4138b3f2c7ea,NaN,[038900029746],[38900029746],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
...,...,...,...,...,...,...,...,...,...
1557097,f993f5e4-422a-4b38-bc49-8906104478b2,NaN,[070462098358],[70462098358],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
1558263,f9c4e2d1-0afb-411f-9c8f-389286d8c61b,NaN,[024000106654],[24000106654],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
1572985,fc1e656e-4082-405b-b66c-ed68beb00a2a,NaN,[024000197409],[24000197409],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2
1577105,fcc9a936-ed1f-406c-8c5e-abe7fcdec6e0,NaN,[038900033125],[38900033125],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2



All consistency validation checks passed.


In [ ]:

import pandas as pd

# ============================================================
# 1. SELECT TWO-DATASET CONFLICTS
# ============================================================

df_leading_zero_check = (
    df_three_way_comparison[
        df_three_way_comparison["consistency_status"]
        == "Two datasets differ; third missing"
    ]
    .copy()
)


# ============================================================
# 2. NORMALIZE EANS FOR COMPARISON ONLY
# ============================================================

def normalize_ean(ean):
    """
    Remove leading zeros only for comparison.

    Examples:
        '024000011859' -> '24000011859'
        '24000011859'  -> '24000011859'

    Original EAN values remain unchanged.
    """
    return str(ean).strip().lstrip("0") or "0"


# ============================================================
# 3. EXTRACT AVAILABLE EANS AND COMPARE
# ============================================================

ean_columns = {
    "Customer": "customer_eans",
    "Pickup": "pickup_eans",
    "Snapshot": "snapshot_eans"
}


def check_leading_zero_difference(row):

    available = {}

    for source, column in ean_columns.items():
        value = row[column]

        if isinstance(value, (list, tuple)) and len(value) == 1:
            available[source] = str(value[0])

    if len(available) != 2:
        return pd.Series({
            "source_1": None,
            "ean_1": None,
            "source_2": None,
            "ean_2": None,
            "leading_zero_only": False,
            "difference_reason": "Unexpected mapping structure"
        })

    (source_1, ean_1), (source_2, ean_2) = list(available.items())

    normalized_1 = normalize_ean(ean_1)
    normalized_2 = normalize_ean(ean_2)

    leading_zero_only = (
        ean_1 != ean_2
        and normalized_1 == normalized_2
    )

    return pd.Series({
        "source_1": source_1,
        "ean_1": ean_1,
        "source_2": source_2,
        "ean_2": ean_2,
        "leading_zero_only": leading_zero_only,
        "difference_reason": (
            "Leading zeros only"
            if leading_zero_only
            else "Different EAN values"
        )
    })


comparison_details = df_leading_zero_check.apply(
    check_leading_zero_difference,
    axis=1
)

df_leading_zero_check = pd.concat(
    [
        df_leading_zero_check,
        comparison_details
    ],
    axis=1
)


# ============================================================
# 4. SEPARATE RESULTS
# ============================================================

df_leading_zero_only = (
    df_leading_zero_check[
        df_leading_zero_check["leading_zero_only"]
    ]
    .copy()
)

df_other_ean_differences = (
    df_leading_zero_check[
        ~df_leading_zero_check["leading_zero_only"]
    ]
    .copy()
)


# ============================================================
# 5. PRINT SUMMARY
# ============================================================

print("=" * 70)
print("LEADING-ZERO ANALYSIS: TWO DATASETS DIFFER")
print("=" * 70)

print(
    "\nTotal two-dataset conflicts:",
    len(df_leading_zero_check)
)

print(
    "Conflicts explained exclusively by leading zeros:",
    len(df_leading_zero_only)
)

print(
    "Conflicts involving genuinely different EAN values:",
    (
        df_leading_zero_check["difference_reason"]
        == "Different EAN values"
    ).sum()
)

print(
    "Cases with unexpected mapping structure:",
    (
        df_leading_zero_check["difference_reason"]
        == "Unexpected mapping structure"
    ).sum()
)

if len(df_leading_zero_check) > 0:
    percentage = (
        len(df_leading_zero_only)
        / len(df_leading_zero_check)
        * 100
    )

    print(
        f"\nPercentage explained by leading zeros: "
        f"{percentage:.2f}%"
    )


# ============================================================
# 6. DISPLAY DETAILED REPORTS
# ============================================================

report_columns = [
    "sku",
    "source_1",
    "ean_1",
    "source_2",
    "ean_2",
    "missing_datasets",
    "leading_zero_only",
    "difference_reason"
]

print("\n=== ALL TWO-DATASET CONFLICTS ===")

display(
    df_leading_zero_check[report_columns]
    .sort_values(["difference_reason", "sku"])
)

print("\n=== CONFLICTS CAUSED ONLY BY LEADING ZEROS ===")

display(
    df_leading_zero_only[report_columns]
    .sort_values("sku")
)

print("\n=== OTHER CONFLICTS ===")

display(
    df_other_ean_differences[report_columns]
    .sort_values("sku")
)


LEADING-ZERO ANALYSIS: TWO DATASETS DIFFER

Total two-dataset conflicts: 127
Conflicts explained exclusively by leading zeros: 110
Conflicts involving genuinely different EAN values: 17
Cases with unexpected mapping structure: 0

Percentage explained by leading zeros: 86.61%

=== ALL TWO-DATASET CONFLICTS ===


,sku,source_1,ean_1,source_2,ean_2,missing_datasets,leading_zero_only,difference_reason
119821,13380c0c-1256-4328-b6ee-e877b71d557f,Pickup,2208262000000,Snapshot,5411863000550,Customer,False,Different EAN values
221211,239278f2-3218-4567-9432-f2ab8edfbbe2,Pickup,9215302933000383000000000340,Snapshot,9215302933000,Customer,False,Different EAN values
380950,3d35104d-6042-4ff7-b6ae-c509a0491bb5,Pickup,9215302933000383000000000340,Snapshot,9215302933000,Customer,False,Different EAN values
480649,4d392d12-1247-47b2-8e71-2ab1ff09c721,Pickup,3257983348552,Snapshot,5400119560986,Customer,False,Different EAN values
498232,4p8mChJJblMIXfuwniZM,Pickup,8710398160829,Snapshot,8710398524089,Customer,False,Different EAN values
...,...,...,...,...,...,...,...,...
1557097,f993f5e4-422a-4b38-bc49-8906104478b2,Pickup,070462098358,Snapshot,70462098358,Customer,True,Leading zeros only
1558263,f9c4e2d1-0afb-411f-9c8f-389286d8c61b,Pickup,024000106654,Snapshot,24000106654,Customer,True,Leading zeros only
1572985,fc1e656e-4082-405b-b66c-ed68beb00a2a,Pickup,024000197409,Snapshot,24000197409,Customer,True,Leading zeros only
1577105,fcc9a936-ed1f-406c-8c5e-abe7fcdec6e0,Pickup,038900033125,Snapshot,38900033125,Customer,True,Leading zeros only



=== CONFLICTS CAUSED ONLY BY LEADING ZEROS ===


,sku,source_1,ean_1,source_2,ean_2,missing_datasets,leading_zero_only,difference_reason
907,0023c4f2-cc0e-4094-8ac0-d647f3c0273d,Pickup,024000017677,Snapshot,24000017677,Customer,True,Leading zeros only
6481,010a9eb8-8427-4948-b986-dc66a606487b,Pickup,038900033125,Snapshot,38900033125,Customer,True,Leading zeros only
14744,025b0474-ee07-4d0d-80d2-f0b3f18765fd,Pickup,031200456991,Snapshot,31200456991,Customer,True,Leading zeros only
19838,032883ef-62e6-42e3-993b-b0fb5eed68bb,Pickup,031200456991,Snapshot,31200456991,Customer,True,Leading zeros only
35922,05bf4207-2c6e-40b0-9c46-4138b3f2c7ea,Pickup,038900029746,Snapshot,38900029746,Customer,True,Leading zeros only
...,...,...,...,...,...,...,...,...
1557097,f993f5e4-422a-4b38-bc49-8906104478b2,Pickup,070462098358,Snapshot,70462098358,Customer,True,Leading zeros only
1558263,f9c4e2d1-0afb-411f-9c8f-389286d8c61b,Pickup,024000106654,Snapshot,24000106654,Customer,True,Leading zeros only
1572985,fc1e656e-4082-405b-b66c-ed68beb00a2a,Pickup,024000197409,Snapshot,24000197409,Customer,True,Leading zeros only
1577105,fcc9a936-ed1f-406c-8c5e-abe7fcdec6e0,Pickup,038900033125,Snapshot,38900033125,Customer,True,Leading zeros only



=== OTHER CONFLICTS ===


,sku,source_1,ean_1,source_2,ean_2,missing_datasets,leading_zero_only,difference_reason
119821,13380c0c-1256-4328-b6ee-e877b71d557f,Pickup,2208262000000,Snapshot,5411863000550,Customer,False,Different EAN values
221211,239278f2-3218-4567-9432-f2ab8edfbbe2,Pickup,9215302933000383000000000340,Snapshot,9215302933000,Customer,False,Different EAN values
380950,3d35104d-6042-4ff7-b6ae-c509a0491bb5,Pickup,9215302933000383000000000340,Snapshot,9215302933000,Customer,False,Different EAN values
480649,4d392d12-1247-47b2-8e71-2ab1ff09c721,Pickup,3257983348552,Snapshot,5400119560986,Customer,False,Different EAN values
498232,4p8mChJJblMIXfuwniZM,Pickup,8710398160829,Snapshot,8710398524089,Customer,False,Different EAN values
503194,50cb9b2c-1527-495e-ad9b-7873866910c0,Pickup,5712840020012,Snapshot,5420036416210,Customer,False,Different EAN values
586227,5e10a675-f170-43ae-bde6-8d7d1c5228b7,Pickup,8710398160829,Snapshot,8710398524089,Customer,False,Different EAN values
783958,7dc71a7a-a7eb-4fc8-b976-aa724f961dd0,Pickup,5410641310157,Snapshot,5410683640960,Customer,False,Different EAN values
933292,95b252dc-5c8e-487b-ab99-5a902a61acec,Pickup,9937237293545905607300187636,Snapshot,20110125005584,Customer,False,Different EAN values
1069822,ab5e5100-296c-422a-8571-120bb5d7cc54,Pickup,3033491279300,Snapshot,8715551422882,Customer,False,Different EAN values


In [29]:

import pandas as pd

# ============================================================
# 1. START FROM THE REMAINING EAN CONFLICTS
# ============================================================

df_conflict_names = (
    df_other_ean_differences[report_columns]
    .sort_values("sku")
    .copy()
)

# ============================================================
# 2. BUILD EAN -> PRODUCT NAME LOOKUPS
# ============================================================

def build_product_name_lookup(df, source_name):

    if "product_name" not in df.columns:
        print(
            f"WARNING: {source_name} does not contain "
            "a 'product_name' column."
        )
        return {}

    names = df[["ean_code", "product_name"]].copy()

    names["ean_code"] = (
        names["ean_code"].astype("string").str.strip()
    )

    names["product_name"] = (
        names["product_name"].astype("string").str.strip()
    )

    names = names.dropna(
        subset=["ean_code", "product_name"]
    )

    names = names[
        (names["ean_code"] != "")
        & (names["product_name"] != "")
    ]

    return (
        names.groupby("ean_code")["product_name"]
        .agg(lambda x: sorted(set(x)))
        .to_dict()
    )


snapshot_name_lookup = build_product_name_lookup(
    df_prod_snap,
    "Snapshot"
)

customer_name_lookup = build_product_name_lookup(
    df_cust_ord,
    "Customer"
)


# ============================================================
# 3. RETRIEVE PRODUCT NAMES FOR BOTH CONFLICTING EANS
# ============================================================

for ean_column in ["ean_1", "ean_2"]:

    df_conflict_names[
        f"{ean_column}_snapshot_names"
    ] = df_conflict_names[ean_column].map(
        lambda ean: snapshot_name_lookup.get(
            str(ean).strip(), []
        )
    )

    df_conflict_names[
        f"{ean_column}_customer_names"
    ] = df_conflict_names[ean_column].map(
        lambda ean: customer_name_lookup.get(
            str(ean).strip(), []
        )
    )


# ============================================================
# 4. ORGANIZE REPORT
# ============================================================

name_report_columns = [
    "sku",
    "source_1",
    "ean_1",
    "ean_1_snapshot_names",
    "ean_1_customer_names",
    "source_2",
    "ean_2",
    "ean_2_snapshot_names",
    "ean_2_customer_names",
    "missing_datasets",
    "difference_reason"
]

df_conflict_names = df_conflict_names[
    name_report_columns
]


# ============================================================
# 5. PRINT SUMMARY
# ============================================================

print("=" * 70)
print("PRODUCT NAME INVESTIGATION FOR CONFLICTING EANS")
print("=" * 70)

print(
    "Total conflicting SKUs investigated:",
    len(df_conflict_names)
)

print(
    "EAN 1 found in snapshot names:",
    df_conflict_names[
        "ean_1_snapshot_names"
    ].map(bool).sum()
)

print(
    "EAN 2 found in snapshot names:",
    df_conflict_names[
        "ean_2_snapshot_names"
    ].map(bool).sum()
)

print(
    "EAN 1 found in customer names:",
    df_conflict_names[
        "ean_1_customer_names"
    ].map(bool).sum()
)

print(
    "EAN 2 found in customer names:",
    df_conflict_names[
        "ean_2_customer_names"
    ].map(bool).sum()
)


# ============================================================
# 6. DISPLAY RESULTS
# ============================================================

display(df_conflict_names)


PRODUCT NAME INVESTIGATION FOR CONFLICTING EANS
Total conflicting SKUs investigated: 17
EAN 1 found in snapshot names: 11
EAN 2 found in snapshot names: 17
EAN 1 found in customer names: 7
EAN 2 found in customer names: 8


,sku,source_1,ean_1,ean_1_snapshot_names,ean_1_customer_names,source_2,ean_2,ean_2_snapshot_names,ean_2_customer_names,missing_datasets,difference_reason
119821,13380c0c-1256-4328-b6ee-e877b71d557f,Pickup,2208262000000,[Burger | Ardennais | Bio],[],Snapshot,5411863000550,[Artisanal | Yaourt brassé | Maigre 0%],[Artisanal | Yaourt brassé | Maigre 0%],Customer,Different EAN values
221211,239278f2-3218-4567-9432-f2ab8edfbbe2,Pickup,9215302933000383000000000340,[],[],Snapshot,9215302933000,[4 Paninis - Jacquets],[],Customer,Different EAN values
380950,3d35104d-6042-4ff7-b6ae-c509a0491bb5,Pickup,9215302933000383000000000340,[],[],Snapshot,9215302933000,[4 Paninis - Jacquets],[],Customer,Different EAN values
480649,4d392d12-1247-47b2-8e71-2ab1ff09c721,Pickup,3257983348552,[Bio Nb Cora Pate Brisee 230Gr],[],Snapshot,5400119560986,[12 Bolo fromage],[12 Bolo fromage],Customer,Different EAN values
498232,4p8mChJJblMIXfuwniZM,Pickup,8710398160829,[Chips nature Oven Baked - LAYS],"[Chips nature Oven Baked - LAY'S, Lay'S Oven N...",Snapshot,8710398524089,[Chips Lays | Naturel 185g],"[Chips Lay's | Naturel 185g, Chips | Naturel]",Customer,Different EAN values
503194,50cb9b2c-1527-495e-ad9b-7873866910c0,Pickup,5712840020012,[Barre | Noix de cajou | Bio],[],Snapshot,5420036416210,[Amandes Brune - Ranobobio],[],Customer,Different EAN values
586227,5e10a675-f170-43ae-bde6-8d7d1c5228b7,Pickup,8710398160829,[Chips nature Oven Baked - LAYS],"[Chips nature Oven Baked - LAY'S, Lay'S Oven N...",Snapshot,8710398524089,[Chips Lays | Naturel 185g],"[Chips Lay's | Naturel 185g, Chips | Naturel]",Customer,Different EAN values
783958,7dc71a7a-a7eb-4fc8-b976-aa724f961dd0,Pickup,5410641310157,[Bister | Moutarde à lancienne],[],Snapshot,5410683640960,[Amandines Bio 5x1p | Tarte à moi],[Amandines Bio 5x1p | Tarte à moi],Customer,Different EAN values
933292,95b252dc-5c8e-487b-ab99-5a902a61acec,Pickup,9937237293545905607300187636,[],[],Snapshot,20110125005584,[Cotes D'Agneau Marinées Celtic],[],Customer,Different EAN values
1069822,ab5e5100-296c-422a-8571-120bb5d7cc54,Pickup,3033491279300,[Danette double saveurs vanille & caramel],[Danette double saveurs vanille & caramel],Snapshot,8715551422882,[AperOH - Chorizo fourré au cheddar],[Aper'OH - Chorizo fourré au cheddar],Customer,Different EAN values


In [31]:

import pandas as pd

# ============================================================
# 1. IDENTIFY DUPLICATE PICKUP KEYS
# ============================================================

duplicate_key = ["order_date", "pickup_point", "sku"]

# All records belonging to duplicate groups
df_pickup_duplicates = (
    df_pickup_corrected[
        df_pickup_corrected.duplicated(
            subset=duplicate_key,
            keep=False
        )
    ]
    .copy()
)

# ============================================================
# 2. SUMMARIZE EACH DUPLICATE GROUP
# ============================================================

df_duplicate_groups = (
    df_pickup_duplicates
    .groupby(duplicate_key, dropna=False)
    .agg(
        number_of_records=("sku", "size"),
        number_of_eans=("ean_code", "nunique"),
        pickup_eans=(
            "ean_code",
            lambda x: sorted(set(x.dropna().astype(str)))
        )
    )
    .reset_index()
)

df_duplicate_groups["has_ean_conflict_within_group"] = (
    df_duplicate_groups["number_of_eans"] > 1
)

# ============================================================
# 3. GET CONFLICTING SKUS FROM PREVIOUS ANALYSIS
# ============================================================

conflicting_skus = set(
    df_other_ean_differences["sku"].dropna().astype(str)
)

df_duplicate_groups["sku_has_cross_dataset_conflict"] = (
    df_duplicate_groups["sku"].astype(str).isin(conflicting_skus)
)

# ============================================================
# 4. FIND OVERLAPPING SKUS
# ============================================================

df_duplicate_conflicting_groups = (
    df_duplicate_groups[
        df_duplicate_groups["sku_has_cross_dataset_conflict"]
    ]
    .copy()
)

df_duplicate_nonconflicting_groups = (
    df_duplicate_groups[
        ~df_duplicate_groups["sku_has_cross_dataset_conflict"]
    ]
    .copy()
)

overlapping_skus = set(
    df_duplicate_conflicting_groups["sku"].astype(str)
)

# Conflicting SKUs without duplicate pickup keys
df_conflicting_without_duplicates = (
    df_other_ean_differences[
        ~df_other_ean_differences["sku"].astype(str).isin(
            overlapping_skus
        )
    ]
    .copy()
)

# ============================================================
# 5. BUILD SKU-LEVEL SUMMARY
# ============================================================

df_duplicate_sku_summary = (
    df_duplicate_groups
    .groupby("sku", dropna=False)
    .agg(
        duplicate_groups=("sku", "size"),
        records_in_duplicate_groups=(
            "number_of_records", "sum"
        ),
        groups_with_different_eans=(
            "has_ean_conflict_within_group", "sum"
        ),
        sku_has_cross_dataset_conflict=(
            "sku_has_cross_dataset_conflict", "max"
        )
    )
    .reset_index()
)

# ============================================================
# 6. PRINT SUMMARY
# ============================================================

print("=" * 75)
print("PICKUP DUPLICATES VS SKU-EAN CONFLICTS")
print("=" * 75)

print(
    "\nTotal duplicate pickup groups:",
    len(df_duplicate_groups)
)

print(
    "Total records belonging to duplicate groups:",
    len(df_pickup_duplicates)
)

print(
    "Distinct SKUs involved in duplicate groups:",
    df_duplicate_groups["sku"].nunique()
)

print(
    "\nTotal distinct conflicting SKUs:",
    len(conflicting_skus)
)

print(
    "Conflicting SKUs also found in duplicate pickup groups:",
    len(overlapping_skus)
)

print(
    "Conflicting SKUs without duplicate pickup groups:",
    len(conflicting_skus - overlapping_skus)
)

print(
    "Duplicate groups containing different EANs:",
    df_duplicate_groups[
        "has_ean_conflict_within_group"
    ].sum()
)

print(
    "Duplicate groups whose SKU also has a cross-dataset conflict:",
    len(df_duplicate_conflicting_groups)
)

print(
    "Duplicate groups with both conditions:",
    df_duplicate_groups[
        (df_duplicate_groups["has_ean_conflict_within_group"])
        & (df_duplicate_groups["sku_has_cross_dataset_conflict"])
    ].shape[0]
)

# ============================================================
# 7. DISPLAY RESULTS
# ============================================================

print("\n=== DUPLICATE GROUPS WITH CONFLICTING SKUS ===")

display(
    df_duplicate_conflicting_groups.sort_values(
        ["sku", "order_date", "pickup_point"]
    )
)

print("\n=== ALL DUPLICATE GROUPS ===")

display(
    df_duplicate_groups.sort_values(
        ["sku", "order_date", "pickup_point"]
    )
)

print("\n=== SKU-LEVEL DUPLICATE SUMMARY ===")

display(
    df_duplicate_sku_summary.sort_values(
        [
            "sku_has_cross_dataset_conflict",
            "duplicate_groups"
        ],
        ascending=[False, False]
    )
)

print("\n=== CONFLICTING SKUS WITHOUT PICKUP DUPLICATES ===")

display(
    df_conflicting_without_duplicates.sort_values("sku")
)

# ============================================================
# 8. OPTIONAL: SHOW ORIGINAL PICKUP RECORDS FOR OVERLAP
# ============================================================

df_overlap_pickup_records = (
    df_pickup_duplicates[
        df_pickup_duplicates["sku"].astype(str).isin(
            overlapping_skus
        )
    ]
    .copy()
)

print("\n=== ACTUAL PICKUP RECORDS FOR OVERLAPPING SKUS ===")

display(
    df_overlap_pickup_records.sort_values(
        duplicate_key
    )
)


PICKUP DUPLICATES VS SKU-EAN CONFLICTS

Total duplicate pickup groups: 14
Total records belonging to duplicate groups: 29
Distinct SKUs involved in duplicate groups: 13

Total distinct conflicting SKUs: 17
Conflicting SKUs also found in duplicate pickup groups: 0
Conflicting SKUs without duplicate pickup groups: 17
Duplicate groups containing different EANs: 0
Duplicate groups whose SKU also has a cross-dataset conflict: 0
Duplicate groups with both conditions: 0

=== DUPLICATE GROUPS WITH CONFLICTING SKUS ===


,order_date,pickup_point,sku,number_of_records,number_of_eans,pickup_eans,has_ean_conflict_within_group,sku_has_cross_dataset_conflict



=== ALL DUPLICATE GROUPS ===


,order_date,pickup_point,sku,number_of_records,number_of_eans,pickup_eans,has_ean_conflict_within_group,sku_has_cross_dataset_conflict
7,2024-05-09,142edab1280b2246,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,3,1,['8719200263796'],False,False
8,2024-05-10,142edab1280b2246,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,2,1,['8719200263796'],False,False
12,2024-05-29,2e815a32c23cc1e2,140ed542-5cb9-4110-aad5-f2036df614ff,2,1,['5400119584227'],False,False
10,2024-05-18,1a1c842a643d33ce,24e3a2e5-f325-4ec9-9fc7-e09e0d828c7c,2,1,['8000430139273'],False,False
0,2024-04-25,2542941672fd21b8,3cf2950b-da59-44ee-b5e9-d5d86aba06e5,2,1,['5400119547581'],False,False
2,2024-05-03,0e7ecfc90eb1dd42,40abbe09-c2d4-47ec-9022-819a6c7fea84,2,1,['5410138051051'],False,False
9,2024-05-16,f8aa98ac894848ac,5a0fc00e-11ee-4adb-9201-3684848e5b85,2,1,['9998883'],False,False
6,2024-05-08,f46feb23542f588a,5fc26cba-9e85-4021-abbd-f02bf06d6255,2,1,['5400119531818'],False,False
3,2024-05-06,f46feb23542f588a,7e0bda49-dec9-4a5d-8c7b-325eb00e85bb,2,1,['8000430139273'],False,False
11,2024-05-23,092a6887b4c84db6,8374d7cd-88ee-4dec-b6d5-815401464181,2,1,['5410146416163'],False,False



=== SKU-LEVEL DUPLICATE SUMMARY ===


,sku,duplicate_groups,records_in_duplicate_groups,groups_with_different_eans,sku_has_cross_dataset_conflict
0,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,2,5,0,False
1,140ed542-5cb9-4110-aad5-f2036df614ff,1,2,0,False
2,24e3a2e5-f325-4ec9-9fc7-e09e0d828c7c,1,2,0,False
3,3cf2950b-da59-44ee-b5e9-d5d86aba06e5,1,2,0,False
4,40abbe09-c2d4-47ec-9022-819a6c7fea84,1,2,0,False
5,5a0fc00e-11ee-4adb-9201-3684848e5b85,1,2,0,False
6,5fc26cba-9e85-4021-abbd-f02bf06d6255,1,2,0,False
7,7e0bda49-dec9-4a5d-8c7b-325eb00e85bb,1,2,0,False
8,8374d7cd-88ee-4dec-b6d5-815401464181,1,2,0,False
9,affd56f3-0a66-4ac1-a07a-82ff0e73c56c,1,2,0,False



=== CONFLICTING SKUS WITHOUT PICKUP DUPLICATES ===


,sku,customer_eans,pickup_eans,snapshot_eans,consistency_status,is_flagged,available_datasets,missing_datasets,number_of_sources,source_1,ean_1,source_2,ean_2,leading_zero_only,difference_reason
119821,13380c0c-1256-4328-b6ee-e877b71d557f,NaN,[2208262000000],[5411863000550],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2,Pickup,2208262000000,Snapshot,5411863000550,False,Different EAN values
221211,239278f2-3218-4567-9432-f2ab8edfbbe2,NaN,[9215302933000383000000000340],[9215302933000],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2,Pickup,9215302933000383000000000340,Snapshot,9215302933000,False,Different EAN values
380950,3d35104d-6042-4ff7-b6ae-c509a0491bb5,NaN,[9215302933000383000000000340],[9215302933000],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2,Pickup,9215302933000383000000000340,Snapshot,9215302933000,False,Different EAN values
480649,4d392d12-1247-47b2-8e71-2ab1ff09c721,NaN,[3257983348552],[5400119560986],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2,Pickup,3257983348552,Snapshot,5400119560986,False,Different EAN values
498232,4p8mChJJblMIXfuwniZM,NaN,[8710398160829],[8710398524089],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2,Pickup,8710398160829,Snapshot,8710398524089,False,Different EAN values
503194,50cb9b2c-1527-495e-ad9b-7873866910c0,NaN,[5712840020012],[5420036416210],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2,Pickup,5712840020012,Snapshot,5420036416210,False,Different EAN values
586227,5e10a675-f170-43ae-bde6-8d7d1c5228b7,NaN,[8710398160829],[8710398524089],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2,Pickup,8710398160829,Snapshot,8710398524089,False,Different EAN values
783958,7dc71a7a-a7eb-4fc8-b976-aa724f961dd0,NaN,[5410641310157],[5410683640960],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2,Pickup,5410641310157,Snapshot,5410683640960,False,Different EAN values
933292,95b252dc-5c8e-487b-ab99-5a902a61acec,NaN,[9937237293545905607300187636],[20110125005584],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2,Pickup,9937237293545905607300187636,Snapshot,20110125005584,False,Different EAN values
1069822,ab5e5100-296c-422a-8571-120bb5d7cc54,NaN,[3033491279300],[8715551422882],Two datasets differ; third missing,True,"Pickup, Snapshot",Customer,2,Pickup,3033491279300,Snapshot,8715551422882,False,Different EAN values



=== ACTUAL PICKUP RECORDS FOR OVERLAPPING SKUS ===


,order_date,pickup_point,sku,ean_code,quantity_ordered,new_price,n_orders


In [32]:

import pandas as pd

# ============================================================
# 1. PREPARE DATASETS
# ============================================================

def prepare_ean_data(df, ean_col="ean_code"):
    result = (
        df[["sku", ean_col]]
        .rename(columns={ean_col: "ean_code"})
        .copy()
    )

    # Keep EANs as strings to preserve leading zeros
    result["sku"] = result["sku"].astype("string").str.strip()
    result["ean_code"] = (
        result["ean_code"].astype("string").str.strip()
    )

    result["sku"] = result["sku"].replace("", pd.NA)
    result["ean_code"] = result["ean_code"].replace("", pd.NA)

    return result


customer = prepare_ean_data(df_cust_ord)

pickup = prepare_ean_data(df_pickup_corrected)

snapshot = prepare_ean_data(
    snapshot_mapping_resolved,
    ean_col="snapshot_ean_code"
)

reference = df_prod_ref[["ean_code"]].copy()

reference["ean_code"] = (
    reference["ean_code"].astype("string").str.strip()
    .replace("", pd.NA)
)


# ============================================================
# 2. BUILD REFERENCE EAN SET
# ============================================================

reference_eans = set(
    reference["ean_code"].dropna()
)

print(
    "Unique valid EANs in product reference:",
    len(reference_eans)
)


# ============================================================
# 3. FIND EANS ABSENT FROM REFERENCE
# ============================================================

def find_missing_reference_eans(df):
    return df.loc[
        df["ean_code"].notna()
        & ~df["ean_code"].isin(reference_eans)
    ].copy()


customer_missing = find_missing_reference_eans(customer)
pickup_missing = find_missing_reference_eans(pickup)
snapshot_missing = find_missing_reference_eans(snapshot)


# ============================================================
# 4. SUMMARY BY DATASET
# ============================================================

datasets = {
    "Customer": customer,
    "Pickup corrected": pickup,
    "Snapshot resolved": snapshot
}

missing_datasets = {
    "Customer": customer_missing,
    "Pickup corrected": pickup_missing,
    "Snapshot resolved": snapshot_missing
}

summary_rows = []

for name, df in datasets.items():

    missing = missing_datasets[name]

    summary_rows.append({
        "dataset": name,
        "total_rows": len(df),
        "rows_with_valid_ean": int(df["ean_code"].notna().sum()),
        "rows_with_null_ean": int(df["ean_code"].isna().sum()),
        "unique_eans": df["ean_code"].nunique(),
        "unique_missing_eans": missing["ean_code"].nunique(),
        "distinct_missing_sku_ean_pairs": len(
            missing[["sku", "ean_code"]].drop_duplicates()
        ),
        "affected_rows": len(missing)
    })

df_missing_ref_summary = pd.DataFrame(summary_rows)


# ============================================================
# 5. BUILD UNIQUE EAN-LEVEL REPORT
# ============================================================

customer_missing_eans = set(
    customer_missing["ean_code"].dropna()
)

pickup_missing_eans = set(
    pickup_missing["ean_code"].dropna()
)

snapshot_missing_eans = set(
    snapshot_missing["ean_code"].dropna()
)

all_missing_eans = (
    customer_missing_eans
    | pickup_missing_eans
    | snapshot_missing_eans
)

df_missing_ref_eans = pd.DataFrame({
    "ean_code": sorted(all_missing_eans)
})

for source, ean_set in [
    ("customer", customer_missing_eans),
    ("pickup", pickup_missing_eans),
    ("snapshot", snapshot_missing_eans)
]:
    df_missing_ref_eans[f"in_{source}"] = (
        df_missing_ref_eans["ean_code"].isin(ean_set)
    )

df_missing_ref_eans["number_of_sources"] = (
    df_missing_ref_eans[
        ["in_customer", "in_pickup", "in_snapshot"]
    ].sum(axis=1)
)


# ============================================================
# 6. ADD RECORD COUNTS PER EAN
# ============================================================

for source, missing in [
    ("customer", customer_missing),
    ("pickup", pickup_missing),
    ("snapshot", snapshot_missing)
]:

    counts = (
        missing.groupby("ean_code")
        .size()
        .rename(f"{source}_records")
        .reset_index()
    )

    df_missing_ref_eans = df_missing_ref_eans.merge(
        counts,
        on="ean_code",
        how="left",
        validate="one_to_one"
    )

record_columns = [
    "customer_records",
    "pickup_records",
    "snapshot_records"
]

df_missing_ref_eans[record_columns] = (
    df_missing_ref_eans[record_columns]
    .fillna(0)
    .astype(int)
)


# ============================================================
# 7. BUILD SKU-EAN-LEVEL REPORT
# ============================================================

customer_pairs = (
    customer_missing[["sku", "ean_code"]]
    .drop_duplicates()
    .assign(source="Customer")
)

pickup_pairs = (
    pickup_missing[["sku", "ean_code"]]
    .drop_duplicates()
    .assign(source="Pickup")
)

snapshot_pairs = (
    snapshot_missing[["sku", "ean_code"]]
    .drop_duplicates()
    .assign(source="Snapshot")
)

all_missing_pairs = pd.concat(
    [customer_pairs, pickup_pairs, snapshot_pairs],
    ignore_index=True
)

df_missing_ref_pairs = (
    all_missing_pairs
    .groupby(["sku", "ean_code"], dropna=False)["source"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="sources")
)

df_missing_ref_pairs["number_of_sources"] = (
    df_missing_ref_pairs["sources"].map(len)
)


# ============================================================
# 8. CLASSIFY SOURCE COVERAGE
# ============================================================

def classify_sources(row):

    sources = []

    if row["in_customer"]:
        sources.append("Customer")

    if row["in_pickup"]:
        sources.append("Pickup")

    if row["in_snapshot"]:
        sources.append("Snapshot")

    return ", ".join(sources)


df_missing_ref_eans["found_in"] = (
    df_missing_ref_eans.apply(
        classify_sources,
        axis=1
    )
)

df_missing_ref_eans = (
    df_missing_ref_eans.sort_values(
        ["number_of_sources", "ean_code"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


# ============================================================
# 9. EXTRACT USEFUL SUBSETS
# ============================================================

# Missing EANs found in all three datasets
df_missing_ref_all_three = (
    df_missing_ref_eans[
        df_missing_ref_eans["number_of_sources"] == 3
    ].copy()
)

# Missing EANs found in exactly two datasets
df_missing_ref_two_sources = (
    df_missing_ref_eans[
        df_missing_ref_eans["number_of_sources"] == 2
    ].copy()
)

# Missing EANs found in only one dataset
df_missing_ref_one_source = (
    df_missing_ref_eans[
        df_missing_ref_eans["number_of_sources"] == 1
    ].copy()
)


# ============================================================
# 10. PRINT SUMMARY
# ============================================================

print("=" * 75)
print("EAN CODES MISSING FROM PRODUCT REFERENCE")
print("=" * 75)

print("\n=== SUMMARY BY DATASET ===")
display(df_missing_ref_summary)

print("\n=== OVERALL SUMMARY ===")

print(
    "Total distinct EANs missing from reference:",
    len(df_missing_ref_eans)
)

print(
    "Distinct SKU-EAN combinations missing from reference:",
    len(df_missing_ref_pairs)
)

print(
    "Missing EANs present in all three datasets:",
    len(df_missing_ref_all_three)
)

print(
    "Missing EANs present in exactly two datasets:",
    len(df_missing_ref_two_sources)
)

print(
    "Missing EANs present in only one dataset:",
    len(df_missing_ref_one_source)
)

print("\n=== MISSING EAN SOURCE DISTRIBUTION ===")

print(
    df_missing_ref_eans["found_in"]
    .value_counts()
    .to_string()
)


# ============================================================
# 11. DISPLAY REPORTS
# ============================================================

print("\n=== ALL EANS MISSING FROM REFERENCE ===")
display(df_missing_ref_eans)

print("\n=== SKU-EAN COMBINATIONS MISSING FROM REFERENCE ===")
display(
    df_missing_ref_pairs.sort_values(
        ["number_of_sources", "sku"],
        ascending=[False, True]
    )
)

print("\n=== MISSING EANS FOUND IN ALL THREE DATASETS ===")
display(df_missing_ref_all_three)

print("\n=== MISSING EANS FOUND IN TWO DATASETS ===")
display(df_missing_ref_two_sources)

print("\n=== MISSING EANS FOUND IN ONLY ONE DATASET ===")
display(df_missing_ref_one_source)


# ============================================================
# 12. VALIDATION
# ============================================================

assert not (
    set(df_missing_ref_eans["ean_code"]) & reference_eans
), "Some reported missing EANs are present in reference."

assert (
    len(df_missing_ref_eans)
    == len(all_missing_eans)
), "Missing EAN count mismatch."

print("\nAll validation checks passed.")


Unique valid EANs in product reference: 63997
EAN CODES MISSING FROM PRODUCT REFERENCE

=== SUMMARY BY DATASET ===


,dataset,total_rows,rows_with_valid_ean,rows_with_null_ean,unique_eans,unique_missing_eans,distinct_missing_sku_ean_pairs,affected_rows
0,Customer,967415,967415,0,37731,2,2,3
1,Pickup corrected,4768673,4768673,0,61049,41,55,282
2,Snapshot resolved,1597714,1597714,0,63995,1,1,1



=== OVERALL SUMMARY ===
Total distinct EANs missing from reference: 41
Distinct SKU-EAN combinations missing from reference: 55
Missing EANs present in all three datasets: 1
Missing EANs present in exactly two datasets: 1
Missing EANs present in only one dataset: 39

=== MISSING EAN SOURCE DISTRIBUTION ===
found_in
Pickup                        39
Customer, Pickup, Snapshot     1
Customer, Pickup               1

=== ALL EANS MISSING FROM REFERENCE ===


,ean_code,in_customer,in_pickup,in_snapshot,number_of_sources,customer_records,pickup_records,snapshot_records,found_in
0,87148111754830,True,True,True,3,2,7,1,"Customer, Pickup, Snapshot"
1,6901668934451,True,True,False,2,1,2,0,"Customer, Pickup"
2,00000000003940,False,True,False,1,0,1,0,Pickup
3,0038900713546,False,True,False,1,0,4,0,Pickup
4,004905570721,False,True,False,1,0,2,0,Pickup
5,0089686120073,False,True,False,1,0,1,0,Pickup
6,016000122222,False,True,False,1,0,15,0,Pickup
7,024000001669,False,True,False,1,0,5,0,Pickup
8,024000106654,False,True,False,1,0,30,0,Pickup
9,034000229154,False,True,False,1,0,2,0,Pickup



=== SKU-EAN COMBINATIONS MISSING FROM REFERENCE ===


,sku,ean_code,sources,number_of_sources
52,fb84e253-caa6-4e94-b9ad-4a91bf33ad31,87148111754830,"[Customer, Pickup, Snapshot]",3
26,8bce0148-e9fd-4489-81da-7cd3a955b377,6901668934451,"[Customer, Pickup]",2
0,0023da91-7ee3-418b-bbab-ab33f31fa090,8711200403271,[Pickup],1
1,08d39d75-2b97-4b8b-8c08-ccb9aaaae321,5400134284287,[Pickup],1
2,08d9def7-5b14-4316-9637-9681102b9e13,997155,[Pickup],1
3,0a2a9736-c1e3-4aa9-ba4c-e68108babe5f,065633471726,[Pickup],1
4,0e89c608-44e0-483d-ab5d-63730100c977,9426617060251,[Pickup],1
5,0f1cf2e7-7cb9-4c8a-b55f-2137840eda6e,9905605601203705603700005586,[Pickup],1
6,11ab5e72-11c8-4b03-a6ed-7326b12a62ed,0038900713546,[Pickup],1
7,14a01028-6e86-425f-86b0-f7967e394d48,024000106654,[Pickup],1


... (3 outputs left out)

,ean_code,in_customer,in_pickup,in_snapshot,number_of_sources,customer_records,pickup_records,snapshot_records,found_in
1,6901668934451,True,True,False,2,1,2,0,"Customer, Pickup"



=== MISSING EANS FOUND IN ONLY ONE DATASET ===


,ean_code,in_customer,in_pickup,in_snapshot,number_of_sources,customer_records,pickup_records,snapshot_records,found_in
2,00000000003940,False,True,False,1,0,1,0,Pickup
3,0038900713546,False,True,False,1,0,4,0,Pickup
4,004905570721,False,True,False,1,0,2,0,Pickup
5,0089686120073,False,True,False,1,0,1,0,Pickup
6,016000122222,False,True,False,1,0,15,0,Pickup
7,024000001669,False,True,False,1,0,5,0,Pickup
8,024000106654,False,True,False,1,0,30,0,Pickup
9,034000229154,False,True,False,1,0,2,0,Pickup
10,038900012250,False,True,False,1,0,9,0,Pickup
11,038900014261,False,True,False,1,0,3,0,Pickup



All validation checks passed.


In [39]:
# from df_missing_ref_one_source add the values on each row of pickup_records, customer_records, and snapshot_records to create a new column called total_missing_records. Then display the updated DataFrame. columns

df_missing_ref_one_source["total_missing_records"]  = df_missing_ref_one_source[["customer_records", "pickup_records", "snapshot_records"]].sum(axis=1)
display(df_missing_ref_one_source)
df_missing_ref_one_source["pickup_total"] = df_missing_ref_one_source["pickup_records"].sum()

display(df_missing_ref_one_source)

,ean_code,in_customer,in_pickup,in_snapshot,number_of_sources,customer_records,pickup_records,snapshot_records,found_in,total_missing_records,pickup_total
2,00000000003940,False,True,False,1,0,1,0,Pickup,1,273
3,0038900713546,False,True,False,1,0,4,0,Pickup,4,273
4,004905570721,False,True,False,1,0,2,0,Pickup,2,273
5,0089686120073,False,True,False,1,0,1,0,Pickup,1,273
6,016000122222,False,True,False,1,0,15,0,Pickup,15,273
7,024000001669,False,True,False,1,0,5,0,Pickup,5,273
8,024000106654,False,True,False,1,0,30,0,Pickup,30,273
9,034000229154,False,True,False,1,0,2,0,Pickup,2,273
10,038900012250,False,True,False,1,0,9,0,Pickup,9,273
11,038900014261,False,True,False,1,0,3,0,Pickup,3,273


,ean_code,in_customer,in_pickup,in_snapshot,number_of_sources,customer_records,pickup_records,snapshot_records,found_in,total_missing_records,pickup_total
2,00000000003940,False,True,False,1,0,1,0,Pickup,1,273
3,0038900713546,False,True,False,1,0,4,0,Pickup,4,273
4,004905570721,False,True,False,1,0,2,0,Pickup,2,273
5,0089686120073,False,True,False,1,0,1,0,Pickup,1,273
6,016000122222,False,True,False,1,0,15,0,Pickup,15,273
7,024000001669,False,True,False,1,0,5,0,Pickup,5,273
8,024000106654,False,True,False,1,0,30,0,Pickup,30,273
9,034000229154,False,True,False,1,0,2,0,Pickup,2,273
10,038900012250,False,True,False,1,0,9,0,Pickup,9,273
11,038900014261,False,True,False,1,0,3,0,Pickup,3,273


In [42]:

import pandas as pd

ERROR_START_DATE = pd.Timestamp("2024-04-01").date()
ERROR_END_DATE = pd.Timestamp("2024-07-31").date()

# ============================================================
# 1. VALIDATE ORIGINAL AND CORRECTED DATAFRAMES
# ============================================================

assert len(df_pickup) == len(df_pickup_corrected), (
    "Original and corrected pickup DataFrames have different row counts."
)

assert df_pickup.index.equals(df_pickup_corrected.index), (
    "Row indices differ. A stable row ID is required to compare corrections."
)

# Verify that the rows still correspond to the same observations
key_columns = ["order_date", "pickup_point", "sku"]

for col in key_columns:
    original_values = df_pickup[col].astype("string").fillna("<NULL>")
    corrected_values = df_pickup_corrected[col].astype("string").fillna("<NULL>")

    assert original_values.equals(corrected_values), (
        f"Column '{col}' differs between original and corrected data."
    )

# ============================================================
# 2. IDENTIFY SKUS WITH MULTIPLE ORIGINAL EANS
# ============================================================

original_ean_counts = (
    df_pickup.assign(
        _ean=df_pickup["ean_code"].astype("string").str.strip()
    )
    .groupby("sku")["_ean"]
    .nunique()
)

ambiguous_pickup_skus = set(
    original_ean_counts[original_ean_counts > 1].index
)

# ============================================================
# 3. IDENTIFY ACTUALLY CORRECTED RECORDS
# ============================================================

original_eans = (
    df_pickup["ean_code"].astype("string").str.strip()
)

corrected_eans = (
    df_pickup_corrected["ean_code"].astype("string").str.strip()
)

# Treat two null values as equal
changed_mask = (
    original_eans.ne(corrected_eans).fillna(False)
    | original_eans.isna().ne(corrected_eans.isna())
)

df_pickup_correction_audit = (
    df_pickup_corrected.loc[changed_mask].copy()
)

df_pickup_correction_audit["old_ean"] = (
    original_eans.loc[changed_mask]
)

df_pickup_correction_audit["corrected_ean"] = (
    corrected_eans.loc[changed_mask]
)

# ============================================================
# 4. CHECK CORRECTION DATES
# ============================================================

df_pickup_correction_audit["audit_date"] = (
    pd.to_datetime(
        df_pickup_correction_audit["order_date"],
        errors="coerce"
    ).dt.date
)

df_pickup_correction_audit["within_error_period"] = (
    df_pickup_correction_audit["audit_date"].between(
        ERROR_START_DATE,
        ERROR_END_DATE,
        inclusive="both"
    )
)

df_pickup_correction_audit["originally_multi_ean_sku"] = (
    df_pickup_correction_audit["sku"].isin(
        ambiguous_pickup_skus
    )
)

# ============================================================
# 5. SEPARATE RESULTS
# ============================================================

df_corrections_inside_period = (
    df_pickup_correction_audit[
        df_pickup_correction_audit["within_error_period"]
    ].copy()
)

df_corrections_outside_period = (
    df_pickup_correction_audit[
        ~df_pickup_correction_audit["within_error_period"]
        & df_pickup_correction_audit["audit_date"].notna()
    ].copy()
)

df_corrections_invalid_dates = (
    df_pickup_correction_audit[
        df_pickup_correction_audit["audit_date"].isna()
    ].copy()
)

# Focus specifically on originally ambiguous SKUs
# whose EANs were corrected outside the error period

df_ambiguous_corrections_outside = (
    df_corrections_outside_period.loc[
        df_corrections_outside_period["originally_multi_ean_sku"]
    ].copy()
)

# ============================================================
# 6. SUMMARY
# ============================================================

print("=" * 75)
print("PICKUP EAN CORRECTION DATE AUDIT")
print("=" * 75)

print(
    "Original SKUs with multiple EANs:",
    len(ambiguous_pickup_skus)
)

print(
    "Total corrected pickup rows:",
    len(df_pickup_correction_audit)
)

print(
    "Corrected rows inside error period:",
    len(df_corrections_inside_period)
)

print(
    "Corrected rows outside error period:",
    len(df_corrections_outside_period)
)

print(
    "Corrected rows with invalid dates:",
    len(df_corrections_invalid_dates)
)

print(
    "Corrections outside period involving originally ambiguous SKUs:",
    len(df_ambiguous_corrections_outside)
)

print(
    "Distinct ambiguous SKUs corrected outside period:",
    df_ambiguous_corrections_outside["sku"].nunique()
)

# ============================================================
# 7. DISPLAY DETAILED REPORTS
# ============================================================

audit_columns = [
    "order_date",
    "pickup_point",
    "sku",
    "old_ean",
    "corrected_ean",
    "originally_multi_ean_sku",
    "within_error_period"
]

# ============================================================
# 7. DISPLAY DETAILED REPORTS
# ============================================================

audit_columns = [
    "order_date",
    "pickup_point",
    "sku",
    "old_ean",
    "corrected_ean",
    "originally_multi_ean_sku",
    "within_error_period"
]

print("\n=== ALL CORRECTIONS OUTSIDE ERROR PERIOD ===")

display(
    df_corrections_outside_period[audit_columns]
    .sort_values(["order_date", "sku"])
)

print("\n=== AMBIGUOUS SKU CORRECTIONS OUTSIDE PERIOD ===")

display(
    df_ambiguous_corrections_outside[audit_columns]
    .sort_values(["order_date", "sku"])
)

print("\n=== ALL CORRECTIONS INSIDE ERROR PERIOD ===")

display(
    df_corrections_inside_period[audit_columns]
    .sort_values(["order_date", "sku"])
)

print("\n=== CORRECTIONS WITH INVALID DATES ===")

display(
    df_corrections_invalid_dates[audit_columns]
)

# ============================================================
# 8. FINAL CHECK
# ============================================================

if len(df_corrections_invalid_dates) > 0:
    print(
        "\nWARNING: Some corrected records have invalid dates. "
        "They must be reviewed before reaching a final conclusion."
    )

if len(df_corrections_outside_period) == 0:
    print(
        "\nRESULT: No corrections with valid dates were made outside "
        "2024-04-01 to 2024-07-31."
    )
else:
    print(
        "\nRESULT: Some EAN corrections were made outside "
        "2024-04-01 to 2024-07-31. Review the rows above."
    )

PICKUP EAN CORRECTION DATE AUDIT
Original SKUs with multiple EANs: 35
Total corrected pickup rows: 53
Corrected rows inside error period: 42
Corrected rows outside error period: 11
Corrected rows with invalid dates: 0
Corrections outside period involving originally ambiguous SKUs: 11
Distinct ambiguous SKUs corrected outside period: 1

=== ALL CORRECTIONS OUTSIDE ERROR PERIOD ===


,order_date,pickup_point,sku,old_ean,corrected_ean,originally_multi_ean_sku,within_error_period
167500,2024-03-06,092a6887b4c84db6,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
167889,2024-03-06,0e7ecfc90eb1dd42,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
168161,2024-03-06,142edab1280b2246,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
168558,2024-03-06,1a1c842a643d33ce,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
168786,2024-03-06,2542941672fd21b8,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
169042,2024-03-06,2e815a32c23cc1e2,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
169245,2024-03-06,48db013424cf43ec,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
169432,2024-03-06,78a2806cc9346b6e,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
169626,2024-03-06,be903990255560d6,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
169859,2024-03-06,cbcb62162e461101,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False



=== AMBIGUOUS SKU CORRECTIONS OUTSIDE PERIOD ===


,order_date,pickup_point,sku,old_ean,corrected_ean,originally_multi_ean_sku,within_error_period
167500,2024-03-06,092a6887b4c84db6,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
167889,2024-03-06,0e7ecfc90eb1dd42,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
168161,2024-03-06,142edab1280b2246,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
168558,2024-03-06,1a1c842a643d33ce,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
168786,2024-03-06,2542941672fd21b8,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
169042,2024-03-06,2e815a32c23cc1e2,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
169245,2024-03-06,48db013424cf43ec,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
169432,2024-03-06,78a2806cc9346b6e,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
169626,2024-03-06,be903990255560d6,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False
169859,2024-03-06,cbcb62162e461101,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,871039852411900100000,871039852411900100,True,False



=== ALL CORRECTIONS INSIDE ERROR PERIOD ===


,order_date,pickup_point,sku,old_ean,corrected_ean,originally_multi_ean_sku,within_error_period
317740,2024-04-25,2542941672fd21b8,3cf2950b-da59-44ee-b5e9-d5d86aba06e5,5411761258817,5400119547581,True,True
334586,2024-04-30,b73fc63876e68b0b,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,3487400006769,8719200263796,True,True
332783,2024-04-30,142edab1280b2246,f0ad65cd-3d4d-4a04-b2d1-53dd637bbced,8710398502230,5410126253993,True,True
341213,2024-05-02,f8aa98ac894848ac,0ab97db6-bd06-4ca5-8fd4-c5fe28e9a5a0,5400119537643,5400113700449,True,True
339941,2024-05-02,b73fc63876e68b0b,d75ccabd-b89c-4f5f-83fd-c2e5415b200a,5400119583169,5410106095698,True,True
337044,2024-05-02,0e7ecfc90eb1dd42,ffd5bb0e-e813-4df1-b4f0-21e2aecfeb2b,2251150000000,3770018934723,True,True
345707,2024-05-03,f46feb23542f588a,1c7dcc4f-590d-4f96-9be8-7a8e8d7b0353,5400112358733,8720600061103,True,True
341821,2024-05-03,0e7ecfc90eb1dd42,1ef44333-2795-4b55-a514-dea3e6582ae5,5410221205453,8002200145491,True,True
341885,2024-05-03,0e7ecfc90eb1dd42,40abbe09-c2d4-47ec-9022-819a6c7fea84,8001100064697,5410138051051,True,True
344387,2024-05-03,78a2806cc9346b6e,c41703dd-e648-4462-a933-3cda0d0000a1,3250392615536,8719500005690,True,True



=== CORRECTIONS WITH INVALID DATES ===


,order_date,pickup_point,sku,old_ean,corrected_ean,originally_multi_ean_sku,within_error_period



RESULT: Some EAN corrections were made outside 2024-04-01 to 2024-07-31. Review the rows above.


In [43]:

import pandas as pd

# ============================================================
# 1. ORIGINAL SNAPSHOT EANS TO INVESTIGATE
# ============================================================

df_old_snapshot_eans = pd.DataFrame([
    {
        "original_sku": "ZasELirVqScsHzHXz9z3",
        "old_snapshot_ean": "3760207115019",
        "corrected_ean": "3760207115033"
    },
    {
        "original_sku": "b8f87d0f-b57e-4c0a-8ff5-0501cbce4277",
        "old_snapshot_ean": "H1556794",
        "corrected_ean": "1556794"
    },
    {
        "original_sku": "fb84e253-caa6-4e94-b9ad-4a91bf33ad31",
        "old_snapshot_ean": "8714811175483",
        "corrected_ean": "87148111754830"
    }
])

# ============================================================
# 2. PREPARE ORIGINAL SNAPSHOT AND TOP DATA
# ============================================================

snap = df_prod_snap.copy()
top = df_prod_top.copy()

for df in [snap, top]:
    df["ean_code"] = (
        df["ean_code"].astype("string").str.strip()
    )

if "sku" in snap.columns:
    snap["sku"] = snap["sku"].astype("string").str.strip()

if "sku" in top.columns:
    top["sku"] = top["sku"].astype("string").str.strip()

# ============================================================
# 3. INVESTIGATE EACH OLD EAN
# ============================================================

results = []
snapshot_matches = []
top_matches = []

for row in df_old_snapshot_eans.itertuples(index=False):

    old_ean = row.old_snapshot_ean
    original_sku = row.original_sku

    # Find every original snapshot record with this EAN
    snap_found = snap.loc[
        snap["ean_code"] == old_ean
    ].copy()

    # Find every product top record with this EAN
    top_found = top.loc[
        top["ean_code"] == old_ean
    ].copy()

    # Distinct SKUs associated with the old EAN
    snapshot_skus = sorted(
        snap_found["sku"].dropna().unique().tolist()
    )

    # SKUs other than the SKU we corrected
    other_snapshot_skus = [
        sku for sku in snapshot_skus
        if sku != original_sku
    ]

    results.append({
        "original_sku": original_sku,
        "old_snapshot_ean": old_ean,
        "corrected_ean": row.corrected_ean,
        "old_ean_still_in_original_snapshots": (
            len(snap_found) > 0
        ),
        "snapshot_records": len(snap_found),
        "snapshot_skus": snapshot_skus,
        "found_under_other_sku": (
            len(other_snapshot_skus) > 0
        ),
        "other_snapshot_skus": other_snapshot_skus,
        "old_ean_in_product_top": len(top_found) > 0,
        "product_top_records": len(top_found)
    })

    if not snap_found.empty:
        snap_found["investigated_old_ean"] = old_ean
        snap_found["investigated_original_sku"] = original_sku
        snapshot_matches.append(snap_found)

    if not top_found.empty:
        top_found["investigated_old_ean"] = old_ean
        top_found["investigated_original_sku"] = original_sku
        top_matches.append(top_found)


# ============================================================
# 4. BUILD REPORTS
# ============================================================

df_old_ean_investigation = pd.DataFrame(results)

df_old_ean_snapshot_records = (
    pd.concat(snapshot_matches, ignore_index=True)
    if snapshot_matches
    else snap.iloc[0:0].copy()
)

df_old_ean_top_records = (
    pd.concat(top_matches, ignore_index=True)
    if top_matches
    else top.iloc[0:0].copy()
)


# ============================================================
# 5. DISPLAY RESULTS
# ============================================================

print("=== OLD SNAPSHOT EAN INVESTIGATION ===")

display(df_old_ean_investigation)

print("\n=== OLD EANS FOUND UNDER OTHER SNAPSHOT SKUS ===")

display(
    df_old_ean_investigation[
        df_old_ean_investigation["found_under_other_sku"]
    ]
)

print("\n=== ORIGINAL SNAPSHOT RECORDS ===")

display(df_old_ean_snapshot_records)

print("\n=== PRODUCT TOP RECORDS WITH OLD EANS ===")

display(df_old_ean_top_records)


=== OLD SNAPSHOT EAN INVESTIGATION ===


,original_sku,old_snapshot_ean,corrected_ean,old_ean_still_in_original_snapshots,snapshot_records,snapshot_skus,found_under_other_sku,other_snapshot_skus,old_ean_in_product_top,product_top_records
0,ZasELirVqScsHzHXz9z3,3760207115019,3760207115033,True,646,"[2300364e-3f15-4f7f-a433-165145ada217, 34517ce...",True,"[2300364e-3f15-4f7f-a433-165145ada217, 34517ce...",True,271
1,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277,H1556794,1556794,True,11,[b8f87d0f-b57e-4c0a-8ff5-0501cbce4277],False,[],True,11
2,fb84e253-caa6-4e94-b9ad-4a91bf33ad31,8714811175483,87148111754830,True,2,"[f18d0aca-ef21-48a2-b004-e99c2d7a50ef, fb84e25...",True,[f18d0aca-ef21-48a2-b004-e99c2d7a50ef],True,1



=== OLD EANS FOUND UNDER OTHER SNAPSHOT SKUS ===


,original_sku,old_snapshot_ean,corrected_ean,old_ean_still_in_original_snapshots,snapshot_records,snapshot_skus,found_under_other_sku,other_snapshot_skus,old_ean_in_product_top,product_top_records
0,ZasELirVqScsHzHXz9z3,3760207115019,3760207115033,True,646,"[2300364e-3f15-4f7f-a433-165145ada217, 34517ce...",True,"[2300364e-3f15-4f7f-a433-165145ada217, 34517ce...",True,271
2,fb84e253-caa6-4e94-b9ad-4a91bf33ad31,8714811175483,87148111754830,True,2,"[f18d0aca-ef21-48a2-b004-e99c2d7a50ef, fb84e25...",True,[f18d0aca-ef21-48a2-b004-e99c2d7a50ef],True,1



=== ORIGINAL SNAPSHOT RECORDS ===


,sku,quantity_sold,unit_weight,new_price,old_price,quantity_end_sales,origin_quantity,quantity_put_on_sale_start_of_day,shop_branch,expiry_date,scan_date,ean_code,snapshot_date,product_name,date_type,category,investigated_old_ean,investigated_original_sku
0,ZasELirVqScsHzHXz9z3,8,0.3,4.2,6.00,275,283,283,90fc7b6861b8d47a,2024-09-01,2024-11-27,3760207115019,2024-11-27,Masque et démélant nourissant huile de coco & ...,DDM,Hygiène,3760207115019,ZasELirVqScsHzHXz9z3
1,yEvO2QRYDM6zeuPSs58U,3,0.3,4.2,6.00,919,922,922,90fc7b6861b8d47a,2049-12-31,2024-11-27,3760207115019,2024-11-27,Masque et démélant nourissant huile de coco & ...,DDM,Hygiène,3760207115019,ZasELirVqScsHzHXz9z3
2,ZasELirVqScsHzHXz9z3,4,0.3,4.2,6.00,267,283,271,90fc7b6861b8d47a,2024-09-01,2024-11-27,3760207115019,2024-11-28,Masque et démélant nourissant huile de coco & ...,DDM,Hygiène,3760207115019,ZasELirVqScsHzHXz9z3
3,yEvO2QRYDM6zeuPSs58U,2,0.3,4.2,6.00,917,922,919,90fc7b6861b8d47a,2049-12-31,2024-11-27,3760207115019,2024-11-28,Masque et démélant nourissant huile de coco & ...,DDM,Hygiène,3760207115019,ZasELirVqScsHzHXz9z3
4,ZasELirVqScsHzHXz9z3,14,0.3,4.2,6.00,260,283,274,90fc7b6861b8d47a,2024-09-01,2024-11-27,3760207115019,2024-11-29,Masque et démélant nourissant huile de coco & ...,DDM,Hygiène,3760207115019,ZasELirVqScsHzHXz9z3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
654,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277,5,0.0,2.0,2.80,1,30,6,2fd94fe219493f93,2026-07-08,2025-05-08,H1556794,2025-05-16,Bière Kasteel tripel,DDM,Boissons,H1556794,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277
655,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277,0,0.0,2.0,2.80,1,30,1,2fd94fe219493f93,2026-07-08,2025-05-08,H1556794,2025-05-17,Bière Kasteel tripel,DDM,Boissons,H1556794,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277
656,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277,1,0.0,2.0,2.80,0,30,1,2fd94fe219493f93,2026-07-08,2025-05-08,H1556794,2025-05-18,Bière Kasteel tripel,DDM,Boissons,H1556794,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277
657,f18d0aca-ef21-48a2-b004-e99c2d7a50ef,1,1.0,2.1,3.49,0,1,1,919fdfbd4f6953e7,2025-11-14,2025-11-07,8714811175483,2025-11-07,Demeter - Kefr BIO,DLC,Boissons,8714811175483,fb84e253-caa6-4e94-b9ad-4a91bf33ad31



=== PRODUCT TOP RECORDS WITH OLD EANS ===


,snapshot_date,ean_code,is_top,investigated_old_ean,investigated_original_sku
0,2024-11-27,3760207115019,False,3760207115019,ZasELirVqScsHzHXz9z3
1,2024-11-28,3760207115019,False,3760207115019,ZasELirVqScsHzHXz9z3
2,2024-11-29,3760207115019,False,3760207115019,ZasELirVqScsHzHXz9z3
3,2024-11-30,3760207115019,False,3760207115019,ZasELirVqScsHzHXz9z3
4,2024-12-01,3760207115019,False,3760207115019,ZasELirVqScsHzHXz9z3
...,...,...,...,...,...
278,2025-05-15,H1556794,None,H1556794,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277
279,2025-05-16,H1556794,None,H1556794,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277
280,2025-05-17,H1556794,None,H1556794,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277
281,2025-05-18,H1556794,None,H1556794,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277


In [44]:
corrected_snapshot_eans = set(
    snapshot_mapping_resolved["snapshot_ean_code"]
    .astype("string")
    .dropna()
    .str.strip()
)

df_old_ean_investigation["still_in_corrected_snapshot_mapping"] = (
    df_old_ean_investigation["old_snapshot_ean"]
    .isin(corrected_snapshot_eans)
)

display(df_old_ean_investigation)

,original_sku,old_snapshot_ean,corrected_ean,old_ean_still_in_original_snapshots,snapshot_records,snapshot_skus,found_under_other_sku,other_snapshot_skus,old_ean_in_product_top,product_top_records,still_in_corrected_snapshot_mapping
0,ZasELirVqScsHzHXz9z3,3760207115019,3760207115033,True,646,"[2300364e-3f15-4f7f-a433-165145ada217, 34517ce...",True,"[2300364e-3f15-4f7f-a433-165145ada217, 34517ce...",True,271,True
1,b8f87d0f-b57e-4c0a-8ff5-0501cbce4277,H1556794,1556794,True,11,[b8f87d0f-b57e-4c0a-8ff5-0501cbce4277],False,[],True,11,False
2,fb84e253-caa6-4e94-b9ad-4a91bf33ad31,8714811175483,87148111754830,True,2,"[f18d0aca-ef21-48a2-b004-e99c2d7a50ef, fb84e25...",True,[f18d0aca-ef21-48a2-b004-e99c2d7a50ef],True,1,True


In [45]:

# Inspect the available columns
print("SNAPSHOT COLUMNS")
print(df_prod_snap.columns.tolist())

print("\nPICKUP COLUMNS")
print(df_pickup_corrected.columns.tolist())

# Inspect likely quantity-related columns
keywords = ["quantity", "qty", "sold", "ordered", "sales", "date"]

for name, df in [
    ("Snapshot", df_prod_snap),
    ("Pickup", df_pickup_corrected)
]:
    print(f"\n{name} candidate columns:")
    print([
        col for col in df.columns
        if any(word in col.lower() for word in keywords)
    ])


SNAPSHOT COLUMNS
['sku', 'quantity_sold', 'unit_weight', 'new_price', 'old_price', 'quantity_end_sales', 'origin_quantity', 'quantity_put_on_sale_start_of_day', 'shop_branch', 'expiry_date', 'scan_date', 'ean_code', 'snapshot_date', 'product_name', 'date_type', 'category']

PICKUP COLUMNS
['order_date', 'pickup_point', 'sku', 'ean_code', 'quantity_ordered', 'new_price', 'n_orders']

Snapshot candidate columns:
['quantity_sold', 'quantity_end_sales', 'origin_quantity', 'quantity_put_on_sale_start_of_day', 'expiry_date', 'scan_date', 'snapshot_date', 'date_type']

Pickup candidate columns:
['order_date', 'quantity_ordered']


In [47]:
import numpy as np
# ============================================================
# 1. CONFIGURATION - ADJUST TO ACTUAL COLUMN NAMES
# ============================================================

SNAPSHOT_DATE_COL = "snapshot_date"
SNAPSHOT_QTY_COL = "quantity_sold"       # Change if needed

PICKUP_DATE_COL = "order_date"
PICKUP_QTY_COL = "quantity_ordered"       # Change if needed

# ============================================================
# 2. PREPARE DATA
# ============================================================

def prepare_daily_quantities(df, date_col, qty_col, source):
    required = ["sku", date_col, qty_col]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            f"{source} missing columns: {missing}. "
            f"Available: {df.columns.tolist()}"
        )

    result = df[required].copy()

    result["sku"] = result["sku"].astype("string").str.strip()
    result["date"] = pd.to_datetime(
        result[date_col], errors="coerce"
    ).dt.normalize()

    result["quantity"] = pd.to_numeric(
        result[qty_col], errors="coerce"
    )

    print(f"\n{source} quality checks:")
    print("Rows:", len(result))
    print("Invalid dates:", result["date"].isna().sum())
    print("Missing SKUs:", result["sku"].isna().sum())
    print("Missing quantities:", result["quantity"].isna().sum())
    print("Negative quantities:", (result["quantity"] < 0).sum())

    # Do not silently convert unknown quantities to zero
    result = result.dropna(
        subset=["date", "sku", "quantity"]
    )

    return result[["date", "sku", "quantity"]]


snapshot_daily_raw = prepare_daily_quantities(
    df_prod_snap,
    SNAPSHOT_DATE_COL,
    SNAPSHOT_QTY_COL,
    "Snapshot"
)

pickup_daily_raw = prepare_daily_quantities(
    df_pickup_corrected,
    PICKUP_DATE_COL,
    PICKUP_QTY_COL,
    "Pickup"
)

# ============================================================
# 3. DETERMINE OVERLAPPING DATE RANGE
# ============================================================

overlap_start = max(
    snapshot_daily_raw["date"].min(),
    pickup_daily_raw["date"].min()
)

overlap_end = min(
    snapshot_daily_raw["date"].max(),
    pickup_daily_raw["date"].max()
)

if pd.isna(overlap_start) or pd.isna(overlap_end):
    raise ValueError("At least one dataset has no valid dated rows.")

if overlap_start > overlap_end:
    raise ValueError("The datasets have no overlapping date range.")

print("\nOVERLAPPING DATE RANGE")
print("Start:", overlap_start.date())
print("End:", overlap_end.date())

snapshot_overlap = snapshot_daily_raw[
    snapshot_daily_raw["date"].between(
        overlap_start, overlap_end
    )
].copy()

pickup_overlap = pickup_daily_raw[
    pickup_daily_raw["date"].between(
        overlap_start, overlap_end
    )
].copy()

# ============================================================
# 4. AGGREGATE TO DATE + SKU
# ============================================================

snapshot_daily = (
    snapshot_overlap
    .groupby(["date", "sku"], as_index=False)
    .agg(
        quantity_sold=("quantity", "sum"),
        snapshot_records=("quantity", "size")
    )
)

pickup_daily = (
    pickup_overlap
    .groupby(["date", "sku"], as_index=False)
    .agg(
        quantity_ordered=("quantity", "sum"),
        pickup_records=("quantity", "size")
    )
)

# ============================================================
# 5. FULL OUTER JOIN
# ============================================================

df_quantity_reconciliation = snapshot_daily.merge(
    pickup_daily,
    on=["date", "sku"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

df_quantity_reconciliation["record_presence"] = (
    df_quantity_reconciliation["_merge"].map({
        "both": "Both datasets",
        "left_only": "Snapshot only",
        "right_only": "Pickup only"
    }).astype("string")
)

# ============================================================
# 6. COMPARE MATCHED QUANTITIES
# ============================================================

matched_mask = (
    df_quantity_reconciliation["_merge"] == "both"
)

df_quantity_reconciliation["quantity_difference"] = (
    df_quantity_reconciliation["quantity_sold"]
    - df_quantity_reconciliation["quantity_ordered"]
)

# Difference = sold - ordered
# Positive: snapshot sold > pickup ordered
# Negative: snapshot sold < pickup ordered

df_quantity_reconciliation["quantities_match"] = pd.Series(
    pd.NA,
    index=df_quantity_reconciliation.index,
    dtype="boolean"
)

df_quantity_reconciliation.loc[
    matched_mask, "quantities_match"
] = np.isclose(
    df_quantity_reconciliation.loc[
        matched_mask, "quantity_sold"
    ].astype(float),
    df_quantity_reconciliation.loc[
        matched_mask, "quantity_ordered"
    ].astype(float),
    rtol=0,
    atol=1e-9
)

df_quantity_reconciliation["reconciliation_status"] = (
    "Only one dataset available"
)

df_quantity_reconciliation.loc[
    matched_mask
    & df_quantity_reconciliation["quantities_match"].fillna(False),
    "reconciliation_status"
] = "Quantities match"

df_quantity_reconciliation.loc[
    matched_mask
    & ~df_quantity_reconciliation["quantities_match"].fillna(False),
    "reconciliation_status"
] = "Quantities differ"

# ============================================================
# 7. CREATE INVESTIGATION SUBSETS
# ============================================================

df_quantity_matches = df_quantity_reconciliation[
    df_quantity_reconciliation["reconciliation_status"]
    == "Quantities match"
].copy()

df_quantity_differences = df_quantity_reconciliation[
    df_quantity_reconciliation["reconciliation_status"]
    == "Quantities differ"
].copy()

df_quantity_unmatched = df_quantity_reconciliation[
    df_quantity_reconciliation["reconciliation_status"]
    == "Only one dataset available"
].copy()

# ============================================================
# 8. SUMMARY
# ============================================================

matched = df_quantity_reconciliation.loc[matched_mask]

print("\n" + "=" * 75)
print("SNAPSHOT VS PICKUP QUANTITY RECONCILIATION")
print("=" * 75)

print("\nDate-SKU combinations:")
print("Total:", len(df_quantity_reconciliation))
print("Present in both:", len(matched))
print("Matching quantities:", len(df_quantity_matches))
print("Different quantities:", len(df_quantity_differences))
print("Only one dataset:", len(df_quantity_unmatched))

if len(matched):
    print(
        "Exact match rate:",
        f"{len(df_quantity_matches) / len(matched):.2%}"
    )

    print(
        "Total snapshot quantity on matched keys:",
        matched["quantity_sold"].sum()
    )

    print(
        "Total pickup quantity on matched keys:",
        matched["quantity_ordered"].sum()
    )

    print(
        "Net quantity difference on matched keys:",
        matched["quantity_difference"].sum()
    )

    print(
        "Total absolute quantity difference:",
        matched["quantity_difference"].abs().sum()
    )

print("\n=== RECONCILIATION STATUS ===")
display(
    df_quantity_reconciliation[
        "reconciliation_status"
    ].value_counts(dropna=False).to_frame("count")
)

print("\n=== QUANTITY DIFFERENCES ===")
display(
    df_quantity_differences.sort_values(
        "quantity_difference",
        key=lambda x: x.abs(),
        ascending=False
    )
)

print("\n=== MATCHING QUANTITIES ===")
display(df_quantity_matches)

print("\n=== UNMATCHED DATE-SKU COMBINATIONS ===")
display(df_quantity_unmatched)

# ============================================================
# 9. VALIDATION
# ============================================================

assert len(df_quantity_reconciliation) == (
    len(df_quantity_matches)
    + len(df_quantity_differences)
    + len(df_quantity_unmatched)
)

print("\nReconciliation audit completed.")



Snapshot quality checks:
Rows: 2977828
Invalid dates: 0
Missing SKUs: 0
Missing quantities: 0
Negative quantities: 224

Pickup quality checks:
Rows: 4768673
Invalid dates: 0
Missing SKUs: 0
Missing quantities: 0
Negative quantities: 0

OVERLAPPING DATE RANGE
Start: 2024-01-02
End: 2026-08-31

SNAPSHOT VS PICKUP QUANTITY RECONCILIATION

Date-SKU combinations:
Total: 2978863
Present in both: 1825184
Matching quantities: 1776692
Different quantities: 48492
Only one dataset: 1153679
Exact match rate: 97.34%
Total snapshot quantity on matched keys: 14393484.0
Total pickup quantity on matched keys: 14327380.0
Net quantity difference on matched keys: 66104.0
Total absolute quantity difference: 91758.0

=== RECONCILIATION STATUS ===


,count
reconciliation_status,
Quantities match,1776692
Only one dataset available,1153679
Quantities differ,48492



=== QUANTITY DIFFERENCES ===


,date,sku,quantity_sold,snapshot_records,quantity_ordered,pickup_records,_merge,record_presence,quantity_difference,quantities_match,reconciliation_status
80014,2024-02-13,62fde5ed-595b-432d-bf24-f00ec48519a1,1294.0,1.0,98.0,10.0,both,Both datasets,1196.0,False,Quantities differ
179833,2024-03-25,45f0afa0-df98-4fc8-a9f3-54d8fa325c22,956.0,1.0,6.0,2.0,both,Both datasets,950.0,False,Quantities differ
2696226,2026-07-07,9e9d34a6-859d-48e4-8c0e-e5cc3159ef19,661.0,1.0,44.0,15.0,both,Both datasets,617.0,False,Quantities differ
2639093,2026-06-25,54ed5d09-eb06-4e3a-9b44-b35a624743ef,1020.0,1.0,490.0,19.0,both,Both datasets,530.0,False,Quantities differ
518753,2024-07-25,5bac0eeb-92d5-4cbb-ae8e-10b4c0f91680,502.0,1.0,2.0,1.0,both,Both datasets,500.0,False,Quantities differ
...,...,...,...,...,...,...,...,...,...,...,...
2978556,2026-08-31,f0c2fa3b-00ce-4abf-8cd1-15371f2af38e,3.0,1.0,2.0,1.0,both,Both datasets,1.0,False,Quantities differ
2978573,2026-08-31,f195b7aa-8560-4dcd-883f-e8e7ddb14c12,7.0,1.0,6.0,4.0,both,Both datasets,1.0,False,Quantities differ
2978613,2026-08-31,f37ab42c-f570-4223-829a-6d01facf4fd6,12.0,1.0,13.0,10.0,both,Both datasets,-1.0,False,Quantities differ
2978247,2026-08-31,e14d9dab-b562-4be7-b6e2-b647b5512c0d,108.0,1.0,107.0,19.0,both,Both datasets,1.0,False,Quantities differ



=== MATCHING QUANTITIES ===


,date,sku,quantity_sold,snapshot_records,quantity_ordered,pickup_records,_merge,record_presence,quantity_difference,quantities_match,reconciliation_status
3,2024-01-02,00555241-28c6-49b5-99c9-6ac6ff33f088,1.0,1.0,1.0,1.0,both,Both datasets,0.0,True,Quantities match
4,2024-01-02,00567e64-f53f-48c8-a4df-16bbecea742f,3.0,1.0,3.0,2.0,both,Both datasets,0.0,True,Quantities match
5,2024-01-02,005f1253-1786-45b2-bb8e-45ef7c3f28f1,6.0,1.0,6.0,2.0,both,Both datasets,0.0,True,Quantities match
6,2024-01-02,007a335b-7425-40ce-90c2-52fc21caee2f,4.0,1.0,4.0,2.0,both,Both datasets,0.0,True,Quantities match
7,2024-01-02,007e4624-f1fb-4580-8618-a5e9bcf95220,3.0,1.0,3.0,1.0,both,Both datasets,0.0,True,Quantities match
...,...,...,...,...,...,...,...,...,...,...,...
2978848,2026-08-31,ffef4f07-069f-4c78-80a4-38668b02c58c,6.0,1.0,6.0,2.0,both,Both datasets,0.0,True,Quantities match
2978850,2026-08-31,fffb5822-2dc5-48a6-ba80-8101104f9133,1.0,1.0,1.0,1.0,both,Both datasets,0.0,True,Quantities match
2978852,2026-08-31,hoUVhnXdDWCAhQVSfn7R,2.0,1.0,2.0,2.0,both,Both datasets,0.0,True,Quantities match
2978858,2026-08-31,rUL4ZT3N6L0QlndlKA35,4.0,1.0,4.0,1.0,both,Both datasets,0.0,True,Quantities match



=== UNMATCHED DATE-SKU COMBINATIONS ===


,date,sku,quantity_sold,snapshot_records,quantity_ordered,pickup_records,_merge,record_presence,quantity_difference,quantities_match,reconciliation_status
0,2024-01-02,003ed021-227b-49d9-b66a-3e0fab41a494,0.0,1.0,NaN,NaN,left_only,Snapshot only,NaN,<NA>,Only one dataset available
1,2024-01-02,004d5850-c4e0-46d5-bbfa-9bf1e6f7287a,0.0,1.0,NaN,NaN,left_only,Snapshot only,NaN,<NA>,Only one dataset available
2,2024-01-02,0054525e-1426-47ec-b664-7e844139e06f,0.0,1.0,NaN,NaN,left_only,Snapshot only,NaN,<NA>,Only one dataset available
29,2024-01-02,030b0880-f4d4-4e28-996d-4675b2676754,0.0,1.0,NaN,NaN,left_only,Snapshot only,NaN,<NA>,Only one dataset available
31,2024-01-02,0352d7ef-a8dd-4830-bab9-3be5c967a0ab,0.0,1.0,NaN,NaN,left_only,Snapshot only,NaN,<NA>,Only one dataset available
...,...,...,...,...,...,...,...,...,...,...,...
2978856,2026-08-31,nobs555ucRYg2A8jzNId,0.0,1.0,NaN,NaN,left_only,Snapshot only,NaN,<NA>,Only one dataset available
2978857,2026-08-31,oUiTgEKFm54eLgokNavC,0.0,1.0,NaN,NaN,left_only,Snapshot only,NaN,<NA>,Only one dataset available
2978859,2026-08-31,s1jPcwp68RkMg6Bwk6uz,0.0,1.0,NaN,NaN,left_only,Snapshot only,NaN,<NA>,Only one dataset available
2978860,2026-08-31,t6aAuShnVmL1adblgFC3,0.0,1.0,NaN,NaN,left_only,Snapshot only,NaN,<NA>,Only one dataset available



Reconciliation audit completed.
